In [3]:
import torch
import os

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())

print("\nKaggle input:")
for root, dirs, files in os.walk("/kaggle/input/competitions/filament-segmentation-2026"):
    level = root.replace("/kaggle/input/competitions/filament-segmentation-2026.", "").count(os.sep)
    if level < 2:
        print(root)

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU count: 2

Kaggle input:


In [4]:
import os

print("=== /kaggle/input ===")
print(os.listdir("/kaggle/input"))

print("\n=== Recursive listing ===")

for root, dirs, files in os.walk("/kaggle/input"):
    print(f"\n[{root}]")
    
    if dirs:
        print("Folders:", dirs[:20])
    
    if files:
        print("Files:", files[:20])

=== /kaggle/input ===
['competitions', 'datasets']

=== Recursive listing ===

[/kaggle/input]
Folders: ['competitions', 'datasets']

[/kaggle/input/competitions]
Folders: ['filament-segmentation-2026']

[/kaggle/input/competitions/filament-segmentation-2026]
Folders: ['MAGFiLO_1.0_Kaggle_2026']

[/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026]
Folders: ['test', 'train']

[/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test]
Folders: ['test_images']

[/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images]
Files: ['20220621185332Bh.jpeg', '20111208173814Mh.jpeg', '20200806072410Th.jpeg', '20111017113714Th.jpeg', '20170926225530Lh.jpeg', '20160327133734Ch.jpeg', '20160820233134Lh.jpeg', '20130310063154Uh.jpeg', '20160828133734Ch.jpeg', '20121203063334Lh.jpeg', '20120813063134Lh.jpeg', '20170902231930Lh.jpeg', '20161219133734Ch.jpeg', '20120111191454Bh.jpeg', '20111015063134Lh.jpeg', '20

In [5]:
import glob

paths = glob.glob("/kaggle/input/**/*", recursive=True)

print("Total paths found:", len(paths))

for p in paths[:100]:
    print(p)

Total paths found: 7484
/kaggle/input/competitions
/kaggle/input/datasets
/kaggle/input/competitions/filament-segmentation-2026
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20220621185332Bh.jpeg
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20111208173814Mh.jpeg
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20200806072410Th.jpeg
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20111017113714Th.jpeg
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.

In [6]:
import os
import json
from pathlib import Path

# ============================================================
# Kaggle competition paths
# ============================================================

DATA_ROOT = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"
)

TRAIN_DIR = DATA_ROOT / "train" / "train_images"
TEST_DIR = DATA_ROOT / "test" / "test_images"
ANNOTATION_FILE = DATA_ROOT / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

print("DATA ROOT:", DATA_ROOT)
print("TRAIN DIR:", TRAIN_DIR)
print("TEST DIR:", TEST_DIR)
print("ANNOTATIONS:", ANNOTATION_FILE)

print("\nPath checks:")
print("Train directory exists:", TRAIN_DIR.exists())
print("Test directory exists:", TEST_DIR.exists())
print("Annotation file exists:", ANNOTATION_FILE.exists())

train_images = list(TRAIN_DIR.glob("*.jpeg"))
test_images = list(TEST_DIR.glob("*.jpeg"))

print("\nDataset counts:")
print("Training images:", len(train_images))
print("Test images:", len(test_images))

DATA ROOT: /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026
TRAIN DIR: /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/train_images
TEST DIR: /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images
ANNOTATIONS: /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json

Path checks:
Train directory exists: True
Test directory exists: True
Annotation file exists: True

Dataset counts:
Training images: 707
Test images: 180


Solar Filament Segmentation — GPU Baseline¶
Dataset
Competition dataset: MAGFiLO 1.0

Image resolution: 2048 × 2048
Training images: 1,154
Ground-truth annotations: MAGFiLO
Test images: competition test set
Baseline
The first model will establish a reproducible semantic-segmentation baseline before introducing instance-aware improvements.

Architecture:

U-Net++ + ResNet34 encoder

Training strategy:

768 × 768 patches
Filament-aware patch sampling
BCE + Dice loss
AdamW optimizer
Mixed precision training
Gradient clipping
Validation on a leakage-safe day-grouped split
The baseline will be evaluated using Dice, IoU and competition-aligned Panoptic Quality (PQ).

In [7]:
import json

with open(ANNOTATION_FILE, "r") as f:
    annotations = json.load(f)

print("Annotation type:", type(annotations))

if isinstance(annotations, dict):
    print("Top-level keys:")
    for key in annotations.keys():
        print(" -", key)

print("\nPreview:")
if isinstance(annotations, dict):
    for key, value in annotations.items():
        if isinstance(value, list):
            print(key, "->", len(value), "items")
        else:
            print(key, "->", type(value).__name__)

Annotation type: <class 'dict'>
Top-level keys:
 - info
 - licenses
 - categories
 - images
 - annotations

Preview:
info -> dict
licenses -> dict
categories -> 4 items
images -> 1154 items
annotations -> 8199 items


In [8]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

if torch.cuda.is_available():
    x = torch.randn(
        2048,
        2048,
        device="cuda"
    )

    y = x @ x.T

    print("\nGPU computation successful.")
    print("Result:", y.shape)

    del x, y
    torch.cuda.empty_cache()

PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

GPU computation successful.
Result: torch.Size([2048, 2048])


In [9]:
from pathlib import Path
import json
from collections import Counter

# Paths
DATA_ROOT = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"
)

TRAIN_DIR = DATA_ROOT / "train" / "train_images"
ANNOTATION_FILE = DATA_ROOT / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

# ------------------------------------------------------------
# 1. Count ALL files in train_images
# ------------------------------------------------------------

all_files = [
    p for p in TRAIN_DIR.rglob("*")
    if p.is_file()
]

print("Total files in train_images:", len(all_files))

extensions = Counter(
    p.suffix.lower()
    for p in all_files
)

print("\nFile extensions:")
for ext, count in extensions.items():
    print(f"{ext}: {count}")

# ------------------------------------------------------------
# 2. Read image filenames from annotation JSON
# ------------------------------------------------------------

with open(ANNOTATION_FILE, "r") as f:
    data = json.load(f)

json_images = data["images"]

json_names = {
    str(img["file_name"])
    for img in json_images
}

print("\nImages listed in JSON:", len(json_names))

# ------------------------------------------------------------
# 3. Compare JSON filenames against actual files
# ------------------------------------------------------------

actual_names = {
    p.name
    for p in all_files
}

missing = sorted(json_names - actual_names)
extra = sorted(actual_names - json_names)

print("JSON images found on disk:", len(json_names & actual_names))
print("Missing from disk:", len(missing))
print("Extra files:", len(extra))

print("\nFirst 20 missing files:")
for name in missing[:20]:
    print(name)

print("\nFirst 20 actual files:")
for name in sorted(actual_names)[:20]:
    print(name)

Total files in train_images: 707

File extensions:
.jpeg: 707

Images listed in JSON: 707
JSON images found on disk: 707
Missing from disk: 0
Extra files: 0

First 20 missing files:

First 20 actual files:
20110109104734Ch.jpeg
20110114105034Ch.jpeg
20110119082634Lh.jpeg
20110123174914Mh.jpeg
20110128174814Mh.jpeg
20110203082634Lh.jpeg
20110205082634Lh.jpeg
20110211084114Th.jpeg
20110213093414Th.jpeg
20110215083814Th.jpeg
20110220082634Lh.jpeg
20110221173814Mh.jpeg
20110301082654Uh.jpeg
20110306082634Lh.jpeg
20110312082634Lh.jpeg
20110313082654Uh.jpeg
20110317082654Uh.jpeg
20110322005814Mh.jpeg
20110324082614Th.jpeg
20110325171014Mh.jpeg


In [10]:
import os

print("=== Kaggle Inputs ===")

for root, dirs, files in os.walk("/kaggle/input/datasets/nytsoul/solar-filament-segmentation"):
    if files:
        print(f"\n{root}")
        for f in files[:10]:
            print("  ", f)

=== Kaggle Inputs ===


Step 2I — Kaggle Project Integration¶
This notebook uses the project code uploaded as a Kaggle Dataset.

The competition data and project code are kept separate conceptually:

Project code: /kaggle/input/datasets/nytsoul/solar-filament-segmentation/
Competition dataset: /kaggle/input/competitions/filament-segmentation-2026/
Training outputs: /kaggle/working/
GPU: Tesla T4 ×2
No files inside /kaggle/input/ will be modified

In [11]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/nytsoul/solar-filament-segmentation/solar-filament-segmentation"
)

COMPETITION_ROOT = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"
)

WORK_ROOT = Path("/kaggle/working/solar-filament-training")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))

print("Project:", PROJECT_ROOT)
print("Project exists:", PROJECT_ROOT.exists())

print("Competition:", COMPETITION_ROOT)
print("Competition exists:", COMPETITION_ROOT.exists())

print("Python path:")
print(sys.path[0])

Project: /kaggle/input/datasets/nytsoul/solar-filament-segmentation/solar-filament-segmentation
Project exists: False
Competition: /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026
Competition exists: True
Python path:
/kaggle/input/datasets/nytsoul/solar-filament-segmentation/solar-filament-segmentation


In [12]:
import sys

sys.path.insert(
    0,
    "/kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation"
)

from src.data.annotations import *
from src.data.masks import *

print("SUCCESS: Antigravity project code imported.")

SUCCESS: Antigravity project code imported.


Step 2J — GPU Training Smoke Test¶
Before full training, verify that the complete pipeline works on Kaggle GPU:

Load a real annotated training image.
Reconstruct its filament mask.
Create a 768×768 training patch.
Initialize U-Net++ + ResNet34.
Run forward pass on GPU.
Calculate BCE + Dice loss.
Run AMP backward pass.
Perform one optimizer update.
Verify gradients, GPU memory and batch timing.
This test must not modify files inside /kaggle/input/

In [13]:
import sys
import time
import torch
from pathlib import Path

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/nytsoul/solar-filament-segmentation/solar-filament-segmentation"
)

sys.path.insert(0, str(PROJECT_ROOT))

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available.")

device = torch.device("cuda")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("\nGPU smoke-test environment ready.")

PyTorch: 2.10.0+cu128
CUDA: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

GPU smoke-test environment ready.


In [14]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation"
)

sys.path.insert(0, str(PROJECT_ROOT))

print("Project:", PROJECT_ROOT)
print("Project exists:", PROJECT_ROOT.exists())

print("\nPyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is not available.")

print("\nKAGGLE GPU: READY")

Project: /kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation
Project exists: True

PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

KAGGLE GPU: READY


In [15]:
SMOKE_TEST = PROJECT_ROOT / "scripts" / "gpu_smoke_test.py"

print(SMOKE_TEST)
print("Exists:", SMOKE_TEST.exists())

/kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation/scripts/gpu_smoke_test.py
Exists: True


In [16]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/nytsoul/solar-filament-segmentation/solar-filament-segmentation"
)

SMOKE_TEST = PROJECT_ROOT / "scripts" / "gpu_smoke_test.py"

print("Project exists:", PROJECT_ROOT.exists())
print("Smoke test exists:", SMOKE_TEST.exists())
print("Smoke test:", SMOKE_TEST)

Project exists: False
Smoke test exists: False
Smoke test: /kaggle/input/datasets/nytsoul/solar-filament-segmentation/solar-filament-segmentation/scripts/gpu_smoke_test.py


# Step 2J — Locate Antigravity Project

Find the actual Kaggle-mounted path of the uploaded Solar Filament Segmentation project.

We must not assume the dataset slug or version suffix.

The project should contain:
- src/
- scripts/
- configs/
- notebooks/
- requirements.txt

After locating it, verify that scripts/gpu_smoke_test.py exists.

In [17]:
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")

print("Searching Kaggle input for the project...\n")

matches = []

for path in KAGGLE_INPUT.rglob("solar-filament-segmentation"):
    if path.is_dir():
        has_src = (path / "src").exists()
        has_scripts = (path / "scripts").exists()
        has_requirements = (path / "requirements.txt").exists()

        if has_src or has_scripts or has_requirements:
            matches.append(path)

if not matches:
    print("PROJECT NOT FOUND")
    print("\nAvailable datasets under /kaggle/input:")
    for p in KAGGLE_INPUT.iterdir():
        print(" -", p)
else:
    print("PROJECT CANDIDATE(S):")
    for p in matches:
        print("\n", p)
        print("  src:", (p / "src").exists())
        print("  scripts:", (p / "scripts").exists())
        print("  requirements.txt:", (p / "requirements.txt").exists())
        print(
            "  gpu_smoke_test.py:",
            (p / "scripts" / "gpu_smoke_test.py").exists()
        )

Searching Kaggle input for the project...

PROJECT CANDIDATE(S):

 /kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation
  src: True
  scripts: True
  requirements.txt: True
  gpu_smoke_test.py: True

 /kaggle/input/datasets/nytsoul/solar-filament-segmentation-2/solar-filament-segmentation
  src: True
  scripts: True
  requirements.txt: True
  gpu_smoke_test.py: True

 /kaggle/input/datasets/nytsoul/solar-filament-segmentation-3/solar-filament-segmentation
  src: True
  scripts: True
  requirements.txt: True
  gpu_smoke_test.py: True

 /kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation
  src: True
  scripts: True
  requirements.txt: True
  gpu_smoke_test.py: True

 /kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation
  src: True
  scripts: True
  requirements.txt: True
  gpu_smoke_test.py: True

 /kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentat

In [18]:
from pathlib import Path

candidates = []

for path in Path("/kaggle/input").rglob("solar-filament-segmentation"):
    if path.is_dir() and (path / "src").exists():
        candidates.append(path)

if not candidates:
    raise FileNotFoundError(
        "Could not find the uploaded Antigravity project under /kaggle/input."
    )

PROJECT_ROOT = candidates[0]

print("PROJECT_ROOT =", PROJECT_ROOT)
print("Project exists:", PROJECT_ROOT.exists())

SMOKE_TEST = PROJECT_ROOT / "scripts" / "gpu_smoke_test.py"

print("Smoke test:", SMOKE_TEST)
print("Smoke test exists:", SMOKE_TEST.exists())

PROJECT_ROOT = /kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation
Project exists: True
Smoke test: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/scripts/gpu_smoke_test.py
Smoke test exists: True


In [19]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available.")

print("\nKAGGLE GPU READY")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

KAGGLE GPU READY


In [20]:
!python "{SMOKE_TEST}"

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
{
  "image_shape": [
    2048,
    2048
  ],
  "raw_mask_shape": [
    2048,
    2048
  ],
  "patch_image_shape": [
    1,
    768,
    768
  ],
  "patch_mask_shape": [
    1,
    768,
    768
  ],
  "mask_statistics": {
    "min": 0.0,
    "max": 1.0,
    "unique_values": [
      0.0,
      1.0
    ]
  },
  "model_parameter_count": 26072337,
  "GPU_name": "Tesla T4",
  "GPU_count": 2,
  "device_used": "cuda",
  "batch_size": 4,
  "loss": 1.6999726295471191,
  "BCE_component": 0.7215415835380554,
  "Dice_component": 0.9784311056137085,
  "batch_time_seconds": 2.998457431793213,
  "peak_GPU_memory_mb": 4849.02587890625,
  "AMP_status": "Enabled",
  "gradient_status": "Finite",
  "status": "PASS"
}

GPU SMOKE TEST: PASS


In [21]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-2/solar-filament-segmentation"
)

print("Project:", PROJECT_ROOT.exists())
print("Training script:", (PROJECT_ROOT / "scripts/train_baseline.py").exists())
print("Split script:", (PROJECT_ROOT / "scripts/kaggle_split.py").exists())

Project: True
Training script: True
Split script: True


In [22]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

CUDA: True
GPUs: 2
0 Tesla T4
1 Tesla T4


In [23]:
!python "{PROJECT_ROOT}/scripts/train_baseline.py"

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
Detected Kaggle environment. Using dataset paths from /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026
2026-09-11 03:34:22,175 [INFO] Using device: cuda
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-2/solar-filament-segmentation/src/data/augmentations.py:33: UserWarning: Argument(s) 'always_apply' are not valid for transform CenterCrop
  A.CenterCrop(height=patch_size, width=patch_size, always_apply=True),
2026-09-11 03:34:23,048 [INFO] Loaded 565 training images and 142 validation images.
2026-09-11 03:34:23,388 [INFO] Using 2 GPUs for DataParallel
2026-09-11 03:34:23,648 [INFO] AMP Status: True
2026-09-11 03:34:23,648 [INFO] Starting training for 30 epochs...
Epoch 1/30 [Val]: 100%|█████████████████████████| 18/18 [00:04<00:00,  4.04i

In [24]:
!python "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-3/solar-filament-segmentation/scripts/evaluate_pq.py"

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
	Missing key(s) in state_dict: "encoder0.0.weight", "encoder0.1.weight", "encoder0.1.bias", "encoder0.1.running_mean", "encoder0.1.running_var", "encoder1.1.0.conv1.weight", "encoder1.1.0.bn1.weight", "encoder1.1.0.bn1.bias", "encoder1.1.0.bn1.running_mean", "encoder1.1.0.bn1.running_var", "encoder1.1.0.conv2.weight", "encoder1.1.0.bn2.weight", "encoder1.1.0.bn2.bias", "encoder1.1.0.bn2.running_mean", "encoder1.1.0.bn2.running_var", "encoder1.1.1.conv1.weight", "encoder1.1.1.bn1.weight", "encoder1.1.1.bn1.bias", "encoder1.1.1.bn1.running_mean", "encoder1.1.1.bn1.running_var", "encoder1.1.1.conv2.weight", "encoder1.1.1.bn2.weight", "encoder1.1.1.bn2.bias", "encoder1.1.1.bn2.running_mean", "encoder1.1.1.bn2.running_var", "encoder1.1.2.conv1.weight", "encoder1.1.2.bn1.weight", "en

In [25]:
!find /kaggle/input -name "evaluate_pq.py" -o -name "best_baseline.pth"

/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/scripts/evaluate_pq.py
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/checkpoints/best_baseline.pth
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-2/solar-filament-segmentation/checkpoints/best_baseline.pth
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-3/solar-filament-segmentation/scripts/evaluate_pq.py
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-3/solar-filament-segmentation/checkpoints/best_baseline.pth
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation/scripts/evaluate_pq.py
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation/checkpoints/best_baseline.pth
/kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation/checkpoints/best_baseline.pth
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-fila

In [26]:
!python /kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/scripts/verify_offline_dependencies.py

--- OFFLINE DEPENDENCY VERIFICATION ---
[OK] torch is available (version: 2.10.0+cu128)
[OK] torchvision is available (version: 0.25.0+cu128)
[OK] timm is available (version: 1.0.26)
[OK] numpy is available (version: 2.0.2)
[OK] pandas is available (version: 2.3.3)
[OK] cv2 is available (version: 4.13.0)
[OK] yaml is available (version: 6.0.3)
[OK] tqdm is available (version: 4.67.3)
[OK] matplotlib is available (version: 3.10.0)

Checking segmentation_models_pytorch...
[INFO] segmentation_models_pytorch not found in system. Checking vendored path: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/vendor
[OK] segmentation_models_pytorch is available via VENDOR (version: 0.5.0)

--- VERIFICATION RESULT ---
ALL REQUIRED DEPENDENCIES ARE AVAILABLE OFFLINE.


In [27]:
!python /kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/scripts/evaluate_pq.py \
  --checkpoint /kaggle/working/checkpoints/best_baseline.pth

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()

--- CHECKPOINT COMPATIBILITY VERIFICATION ---
Checkpoint Path: /kaggle/working/checkpoints/best_baseline.pth
Model Class: UnetPlusPlus
Encoder: ResNetEncoder
Parameter Count: 26,072,337
Detected DataParallel checkpoint (module. prefix). Stripping prefix...
Number of Checkpoint Keys: 340
Number of Model Keys: 350
Missing Keys: 292
  - encoder.conv1.weight
  - encoder.bn1.weight
  - encoder.bn1.bias
  - encoder.bn1.running_mean
  - encoder.bn1.running_var
  ... and 287 more
Unexpected Keys: 340
  - encoder0.0.weight
  - encoder0.1.weight
  - encoder0.1.bias
  - encoder0.1.running_mean
  - encoder0.1.running_var
  ... and 335 more
Traceback (most recent call last):
  File "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/scripts/evaluate_pq

In [28]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "best_baseline.pth":
            print(os.path.join(root, f))

/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/checkpoints/best_baseline.pth
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-2/solar-filament-segmentation/checkpoints/best_baseline.pth
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-3/solar-filament-segmentation/checkpoints/best_baseline.pth
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation/checkpoints/best_baseline.pth
/kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation/checkpoints/best_baseline.pth
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/checkpoints/best_baseline.pth


In [29]:
!find /kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation -maxdepth 2 -type f | head -30

/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/DATASET.md
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/outputs/train_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/outputs/kaggle_val_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/outputs/kaggle_train_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/outputs/kaggle_split_analysis.md
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/outputs/training_curves.png
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/outputs/final_metrics.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/outputs/kaggle_integration_report.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar

In [30]:
!python /kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/scripts/verify_offline_dependencies.py

--- OFFLINE DEPENDENCY VERIFICATION ---
[OK] torch is available (version: 2.10.0+cu128)
[OK] torchvision is available (version: 0.25.0+cu128)
[OK] timm is available (version: 1.0.26)
[OK] numpy is available (version: 2.0.2)
[OK] pandas is available (version: 2.3.3)
[OK] cv2 is available (version: 4.13.0)
[OK] yaml is available (version: 6.0.3)
[OK] tqdm is available (version: 4.67.3)
[OK] matplotlib is available (version: 3.10.0)

Checking segmentation_models_pytorch...
[INFO] segmentation_models_pytorch not found in system. Checking vendored path: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/vendor
[OK] segmentation_models_pytorch is available via VENDOR (version: 0.5.0)

--- VERIFICATION RESULT ---
ALL REQUIRED DEPENDENCIES ARE AVAILABLE OFFLINE.


In [31]:
!python /kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/scripts/evaluate_pq.py \
  --checkpoint /kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/checkpoints/best_baseline.pth

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()

--- CHECKPOINT COMPATIBILITY VERIFICATION ---
Checkpoint Path: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/checkpoints/best_baseline.pth
Model Class: UnetPlusPlus
Encoder: ResNetEncoder
Parameter Count: 26,072,337
Number of Checkpoint Keys: 350
Number of Model Keys: 350
Missing Keys: 0
Unexpected Keys: 0
Successful Strict Loading Status: YES
------------------------------------------

Evaluating PQ: 100%|██████████████████████████| 142/142 [02:21<00:00,  1.01it/s]
Generating visualizations for representative images...

BASELINE PQ EVALUATION
----------------------
Validation images: 142
Dice: 0.0948
IoU: 0.0530
PQ: 0.0005
SQ: 0.0039
RQ: 0.0009
TP: 1
FP: 811
FN: 1008
Fragmentation rate: 0.0168
Merging rate: 0.0049
Average predicted i

In [32]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

PyTorch: 2.10.0+cu128
CUDA: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [33]:
!nvidia-smi

Fri Sep 11 04:03:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             33W /   70W |     137MiB /  15360MiB |     28%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [34]:
!python -c "import torch; print(torch.__version__); print(torch.cuda.is_available())"


2.10.0+cu128
True


In [1]:
!python /kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/scripts/train_baseline_v2.py \
    --config /kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/configs/baseline_v2.yaml

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
Detected Kaggle environment. Using dataset paths from /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026
2026-09-11 16:04:46,793 [INFO] Using device: cuda
2026-09-11 16:04:48,202 [INFO] Training patches per epoch: 2260 (565 images x 4 patches)
2026-09-11 16:04:48,203 [INFO] Validation images: 142 (full 2048x2048)
2026-09-11 16:04:48,641 [INFO] Using 2 GPUs for DataParallel
2026-09-11 16:04:48,965 [INFO] AMP Status: True
2026-09-11 16:04:48,965 [INFO] Sampling probabilities: filament=0.50, disk=0.20, limb=0.15, background=0.15
2026-09-11 16:04:48,965 [INFO] Starting training for 40 epochs...
Epoch 1/40 [Train]: 100%|█████████████████████| 565/565 [04:17<00:00,  2.20it/s]
2026-09-11 16:09:06,147 [INFO]   Sampling: filament=49.9%, disk=19.7%, limb=15.8%,

IOStream.flush timed out


In [11]:
import json
import os

for path in [
    "/kaggle/working/outputs/full_image_baseline_v2/final_metrics_v2.json",
    "/kaggle/working/outputs/full_image_baseline_v2/training_history_v2.json",
]:
    print("\n", path, os.path.exists(path))
    if os.path.exists(path):
        print(open(path).read()[:3000])


 /kaggle/working/outputs/full_image_baseline_v2/final_metrics_v2.json False

 /kaggle/working/outputs/full_image_baseline_v2/training_history_v2.json False


In [17]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        if "baseline_v2" in f.lower() or f.endswith(".pth"):
            path = os.path.join(root, f)
            print(path, f"{os.path.getsize(path)/(1024**2):.1f} MB")

In [18]:
import os

print("=== V2 CHECKPOINT SEARCH ===")

for root, dirs, files in os.walk("/kaggle"):
    for f in files:
        if f.endswith(".pth") and ("v2" in f.lower() or "baseline" in f.lower()):
            path = os.path.join(root, f)
            print(path, f"{os.path.getsize(path)/(1024**2):.1f} MB")

=== V2 CHECKPOINT SEARCH ===
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-3/solar-filament-segmentation/checkpoints/best_baseline.pth 99.7 MB
/kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation/checkpoints/best_baseline.pth 99.7 MB
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation/checkpoints/best_baseline.pth 99.7 MB
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/checkpoints/best_baseline.pth 99.7 MB
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/checkpoints/latest_checkpoint_v2.pth 99.7 MB
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/checkpoints/best_baseline_v2.pth 99.7 MB
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/checkpoints/best_baseline.pth 99.7 MB


In [19]:
import os

ckpt = "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/checkpoints/best_baseline_v2.pth"

print("Exists:", os.path.isfile(ckpt))
print("Size:", round(os.path.getsize(ckpt)/(1024**2), 2), "MB")

Exists: True
Size: 99.66 MB


In [1]:
!python /kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/scripts/evaluate_pq.py \
    --checkpoint /kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/checkpoints/best_baseline_v2.pth

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()

--- CHECKPOINT COMPATIBILITY VERIFICATION ---
Checkpoint Path: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/checkpoints/best_baseline_v2.pth
Model Class: UnetPlusPlus
Encoder: ResNetEncoder
Parameter Count: 26,072,337
Number of Checkpoint Keys: 350
Number of Model Keys: 350
Missing Keys: 0
Unexpected Keys: 0
Successful Strict Loading Status: YES
------------------------------------------

Evaluating PQ: 100%|██████████████████████████| 142/142 [07:01<00:00,  2.97s/it]
Generating visualizations for representative images...

BASELINE PQ EVALUATION
----------------------
Validation images: 142
Dice: 0.0111
IoU: 0.0056
PQ: 0.0000
SQ: 0.0000
RQ: 0.0000
TP: 0
FP: 86968
FN: 1009
Fragmentation rate: 0.3548
Merging rate: 0.0009
Average predic

In [3]:
!python /kaggle/input/datasets/nytsoul/solar-filament-segmentation-7/solar-filament-segmentation/scripts/evaluate_pq.py \
  --checkpoint /kaggle/input/datasets/nytsoul/solar-filament-segmentation-7/solar-filament-segmentation/checkpoints/best_baseline_v2.pth

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()

--- CHECKPOINT COMPATIBILITY VERIFICATION ---
Checkpoint Path: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-7/solar-filament-segmentation/checkpoints/best_baseline_v2.pth
Model Class: UnetPlusPlus
Encoder: ResNetEncoder
Parameter Count: 26,072,337
Number of Checkpoint Keys: 350
Number of Model Keys: 350
Missing Keys: 0
Unexpected Keys: 0
Successful Strict Loading Status: YES
------------------------------------------

Evaluating PQ: 100%|██████████████████████████| 142/142 [06:57<00:00,  2.94s/it]
Generating visualizations for representative images...

BASELINE PQ EVALUATION
----------------------
Validation images: 142
Dice: 0.0112
IoU: 0.0057
PQ: 0.0000
SQ: 0.0000
RQ: 0.0000
TP: 0
FP: 86968
FN: 1009
Fragmentation rate: 0.3548
Merging rate: 0.0009
Average predic

In [4]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

CUDA: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [6]:
!pwd
!ls -la
!find /kaggle/working -maxdepth 3 -type f | grep -E "baseline_v3|train_baseline_v3"

/kaggle/working
total 12
drwxr-xr-x 3 root root 4096 Sep 22 16:59 .
drwxr-xr-x 5 root root 4096 Sep 22 16:59 ..
drwxr-xr-x 2 root root 4096 Sep 22 16:59 .virtual_documents


In [8]:
!find /kaggle/input -type f \( -name "train_baseline_v3.py" -o -name "baseline_v3.yaml" \)

/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/scripts/train_baseline_v3.py
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/configs/baseline_v3.yaml


In [10]:
!cp -r /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/* /kaggle/working/

In [11]:
!ls -lh /kaggle/working/scripts/train_baseline_v3.py
!ls -lh /kaggle/working/configs/baseline_v3.yaml

-rw-r--r-- 1 root root 23K Sep 22 17:04 /kaggle/working/scripts/train_baseline_v3.py
-rw-r--r-- 1 root root 1.4K Sep 22 17:04 /kaggle/working/configs/baseline_v3.yaml


In [12]:
!python /kaggle/working/scripts/train_baseline_v3.py --help

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
usage: train_baseline_v3.py [-h] [--config CONFIG]
                            [--limit_batches LIMIT_BATCHES] [--epochs EPOCHS]
                            [--out_dir OUT_DIR] [--val_limit VAL_LIMIT]

options:
  -h, --help            show this help message and exit
  --config CONFIG
  --limit_batches LIMIT_BATCHES
  --epochs EPOCHS
  --out_dir OUT_DIR
  --val_limit VAL_LIMIT
                        Limit number of validation images for smoke testing


In [13]:
!python /kaggle/working/scripts/train_baseline_v3.py \
    --config /kaggle/working/configs/baseline_v3.yaml \
    --epochs 10

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
Detected Kaggle environment. Using dataset paths from /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026
2026-09-22 17:07:32,455 [INFO] Using device: cuda
2026-09-22 17:07:33,896 [INFO] Training patches per epoch: 2260 (565 images x 4 patches)
2026-09-22 17:07:33,896 [INFO] Validation images: 142 (full 2048x2048)
2026-09-22 17:07:34,347 [INFO] Initializing final segmentation_head bias to -4.5951 for prior 0.01
2026-09-22 17:07:34,996 [INFO] Initial model mean probability (on zero input): 0.0100
2026-09-22 17:07:35,027 [INFO] Using 2 GPUs for DataParallel
2026-09-22 17:07:35,372 [INFO] Loss config: Focal(alpha=0.25, gamma=2.0) + Tversky(alpha=0.7, beta=0.3)
2026-09-22 17:07:35,373 [INFO] AMP Status: True
2026-09-22 17:07:35,373 [INFO] Sampling probabil

In [15]:
!python /kaggle/working/scripts/evaluate_pq.py \
    --checkpoint /kaggle/working/checkpoints/best_baseline_v3.pth

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()

--- CHECKPOINT COMPATIBILITY VERIFICATION ---
Checkpoint Path: /kaggle/working/checkpoints/best_baseline_v3.pth
Model Class: UnetPlusPlus
Encoder: ResNetEncoder
Parameter Count: 26,072,337
Detected DataParallel checkpoint (module. prefix). Stripping prefix...
Number of Checkpoint Keys: 350
Number of Model Keys: 350
Missing Keys: 0
Unexpected Keys: 0
Successful Strict Loading Status: YES
------------------------------------------

Evaluating PQ: 100%|██████████████████████████| 142/142 [02:17<00:00,  1.03it/s]
Generating visualizations for representative images...

BASELINE PQ EVALUATION
----------------------
Validation images: 142
Dice: 0.5810
IoU: 0.4313
PQ: 0.2819
SQ: 0.6136
RQ: 0.4191
TP: 513
FP: 955
FN: 496
Fragmentation rate: 0.1388
Merging rate: 0.0048
Average predicted

In [19]:
!python scripts/diagnose_thresholds.py --limit 142

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
Processing images: 100%|██████████████████████| 142/142 [10:56<00:00,  4.62s/it]

--- Threshold Diagnostic Summary ---
 threshold  mean_dice  mean_iou  mean_fg_pct  mean_pred_cc  mean_gt_cc  mean_tp    mean_fp  mean_fn
      0.10   0.007592  0.003819   100.000000      1.000000    7.105634      0.0   1.000000 7.105634
      0.20   0.007592  0.003819   100.000000      1.000000    7.105634      0.0   1.000000 7.105634
      0.30   0.007592  0.003819    99.998465      1.000000    7.105634      0.0   1.000000 7.105634
      0.40   0.007767  0.003908    97.731733      6.105634    7.105634      0.0   6.105634 7.105634
      0.45   0.009088  0.004578    83.288375     77.767606    7.105634      0.0  77.767606 7.105634
      0.50   0.011221  0.005663    22.860204    612.450704    7.10563

In [21]:
!grep -n -E "CHECKPOINT|checkpoint|best_baseline|\.pth" scripts/diagnose_thresholds.py

53:    checkpoint_path = os.path.join(base_dir, 'checkpoints', 'best_baseline_v2.pth')
54:    if not os.path.exists(checkpoint_path):
55:        print(f"Warning: local checkpoint {checkpoint_path} not found. Trying Kaggle path.")
56:        checkpoint_path = '/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/checkpoints/best_baseline_v2.pth'
58:    state_dict = torch.load(checkpoint_path, map_location=device)


In [28]:
!cp /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/scripts/diagnose_thresholds.py /kaggle/working/diagnose_thresholds.py

In [29]:
path = "/kaggle/working/diagnose_thresholds.py"

with open(path, "r") as f:
    code = f.read()

code = code.replace(
    "checkpoints/best_baseline_v2.pth",
    "checkpoints/best_baseline_v3.pth"
)

code = code.replace(
    "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/checkpoints/best_baseline_v2.pth",
    "/kaggle/working/checkpoints/best_baseline_v3.pth"
)

with open(path, "w") as f:
    f.write(code)

print("V2 → V3 checkpoint updated")

V2 → V3 checkpoint updated


In [30]:
!grep -n -E "checkpoint|best_baseline|pth" /kaggle/working/diagnose_thresholds.py

53:    checkpoint_path = os.path.join(base_dir, 'checkpoints', 'best_baseline_v2.pth')
54:    if not os.path.exists(checkpoint_path):
55:        print(f"Warning: local checkpoint {checkpoint_path} not found. Trying Kaggle path.")
56:        checkpoint_path = '/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/checkpoints/best_baseline_v3.pth'
58:    state_dict = torch.load(checkpoint_path, map_location=device)


In [33]:
path = "/kaggle/working/diagnose_thresholds.py"

with open(path, "r") as f:
    code = f.read()

start = code.index("    checkpoint_path = os.path.join")
end = code.index("    state_dict = torch.load", start)

replacement = '''    checkpoint_path = "/kaggle/working/checkpoints/best_baseline_v3.pth"
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"V3 checkpoint not found: {checkpoint_path}"
        )
    print(f"Using V3 checkpoint: {checkpoint_path}")
'''

code = code[:start] + replacement + code[end:]

with open(path, "w") as f:
    f.write(code)

print("DONE")

DONE


In [34]:
!sed -n '48,62p' /kaggle/working/diagnose_thresholds.py

    )
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = get_baseline_model(config).to(device)
    
    checkpoint_path = "/kaggle/working/checkpoints/best_baseline_v3.pth"
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"V3 checkpoint not found: {checkpoint_path}"
        )
    print(f"Using V3 checkpoint: {checkpoint_path}")
    state_dict = torch.load(checkpoint_path, map_location=device)
    if list(state_dict.keys())[0].startswith('module.'):
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=True)


In [36]:
!python /kaggle/working/diagnose_thresholds.py --limit 5

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
Traceback (most recent call last):
  File "/kaggle/working/diagnose_thresholds.py", line 182, in <module>
    main()
  File "/kaggle/working/diagnose_thresholds.py", line 29, in main
    with open(config_path, 'r') as f:
         ^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/configs/baseline_v2.yaml'


In [40]:
!cp /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/configs/baseline_v3.yaml /kaggle/working/
!cp /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/scripts/diagnose_thresholds.py /kaggle/working/

In [42]:
!python /kaggle/working/diagnose_thresholds.py \
    --config /kaggle/working/baseline_v3.yaml

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
usage: diagnose_thresholds.py [-h] [--limit LIMIT]
diagnose_thresholds.py: error: unrecognized arguments: --config /kaggle/working/baseline_v3.yaml


In [41]:
!ls -lh /kaggle/working/baseline_v3.yaml
!ls -lh /kaggle/working/diagnose_thresholds.py

-rw-r--r-- 1 root root 1.4K Sep 22 18:48 /kaggle/working/baseline_v3.yaml
-rw-r--r-- 1 root root 7.0K Sep 22 18:48 /kaggle/working/diagnose_thresholds.py


In [44]:
!sed -i 's|/kaggle/configs/baseline_v2.yaml|/kaggle/working/baseline_v3.yaml|g' /kaggle/working/diagnose_thresholds.py

In [45]:
!grep -n "config_path\|checkpoint_path\|baseline_v" /kaggle/working/diagnose_thresholds.py

28:    config_path = os.path.join(base_dir, 'configs', 'baseline_v2.yaml')
29:    with open(config_path, 'r') as f:
53:    checkpoint_path = os.path.join(base_dir, 'checkpoints', 'best_baseline_v2.pth')
54:    if not os.path.exists(checkpoint_path):
55:        print(f"Warning: local checkpoint {checkpoint_path} not found. Trying Kaggle path.")
56:        checkpoint_path = '/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/checkpoints/best_baseline_v2.pth'
58:    state_dict = torch.load(checkpoint_path, map_location=device)


In [46]:
!sed -i "s|config_path = os.path.join(base_dir, 'configs', 'baseline_v2.yaml')|config_path = '/kaggle/working/baseline_v3.yaml'|" /kaggle/working/diagnose_thresholds.py

!sed -i "s|checkpoint_path = os.path.join(base_dir, 'checkpoints', 'best_baseline_v2.pth')|checkpoint_path = '/kaggle/working/checkpoints/best_baseline_v3.pth'|" /kaggle/working/diagnose_thresholds.py

!sed -i "s|/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/checkpoints/best_baseline_v2.pth|/kaggle/working/checkpoints/best_baseline_v3.pth|" /kaggle/working/diagnose_thresholds.py

In [47]:
!grep -n "config_path\|checkpoint_path\|baseline_v" /kaggle/working/diagnose_thresholds.py

28:    config_path = '/kaggle/working/baseline_v3.yaml'
29:    with open(config_path, 'r') as f:
53:    checkpoint_path = '/kaggle/working/checkpoints/best_baseline_v3.pth'
54:    if not os.path.exists(checkpoint_path):
55:        print(f"Warning: local checkpoint {checkpoint_path} not found. Trying Kaggle path.")
56:        checkpoint_path = '/kaggle/working/checkpoints/best_baseline_v3.pth'
58:    state_dict = torch.load(checkpoint_path, map_location=device)


In [48]:
!python /kaggle/working/diagnose_thresholds.py --limit 142

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
Traceback (most recent call last):
  File "/kaggle/working/diagnose_thresholds.py", line 181, in <module>
    main()
  File "/kaggle/working/diagnose_thresholds.py", line 43, in main
    val_dataset = MAGFiLOFullImageDataset(
                  ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/src/data/dataset_fullimg.py", line 35, in __init__
    self.df = pd.read_csv(csv_path)
              ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^

In [49]:
!find /kaggle/input -name "kaggle_val_split.csv" -type f

/kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation/outputs/kaggle_val_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-7/solar-filament-segmentation/outputs/kaggle_val_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-7/solar-filament-segmentation/outputs/full_image_baseline_v2/kaggle_val_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-3/solar-filament-segmentation/outputs/kaggle_val_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/outputs/kaggle_val_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/outputs/full_image_baseline_v2/kaggle_val_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation/outputs/kaggle_val_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/outputs/kaggle_val_split.csv
/kaggle/inpu

In [50]:
!find /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9 -name "*.csv" -type f

/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/train_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_train_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/full_image_baseline_v2/kaggle_val_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/full_image_baseline_v2/kaggle_train_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/full_image_baseline_v2/training_history_v2.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/full_image_baseline_v2/val_limited.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentat

In [51]:
!grep -n "kaggle_val_split\|csv_path" /kaggle/working/diagnose_thresholds.py

38:    val_csv_path = os.path.join(base_dir, 'outputs', 'kaggle_val_split.csv')
44:        csv_path=val_csv_path,


In [53]:
csv_path = "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv"

In [57]:
!grep -n "kaggle_val_split\|csv_path" /kaggle/working/diagnose_thresholds.py

38:    val_csv_path = os.path.join(base_dir, 'outputs', 'kaggle_val_split.csv')
44:        csv_path=val_csv_path,


In [61]:
!sed -i "s|os.path.join(base_dir, 'outputs', 'kaggle_val_split.csv')|'/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv'|" /kaggle/working/diagnose_thresholds.py

In [62]:
!sed -i "s|/kaggle/outputs/kaggle_val_split.csv|/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv|g" /kaggle/working/diagnose_thresholds.py

In [63]:
!grep -n "config_path\|checkpoint_path\|kaggle_val_split" /kaggle/working/diagnose_thresholds.py

28:    config_path = '/kaggle/working/baseline_v3.yaml'
29:    with open(config_path, 'r') as f:
38:    val_csv_path = '/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv'
53:    checkpoint_path = '/kaggle/working/checkpoints/best_baseline_v3.pth'
54:    if not os.path.exists(checkpoint_path):
55:        print(f"Warning: local checkpoint {checkpoint_path} not found. Trying Kaggle path.")
56:        checkpoint_path = '/kaggle/working/checkpoints/best_baseline_v3.pth'
58:    state_dict = torch.load(checkpoint_path, map_location=device)


In [64]:
import pandas as pd

csv_path = "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv"

df = pd.read_csv(csv_path)

print("CSV loaded:", True)
print("Rows:", len(df))
print(df.head())

CSV loaded: True
Rows: 142
                  image_id              file_name         day  num_filaments
0  010102-20141027025054Uh  20141027025054Uh.jpeg  2014-10-27             14
1  010201-20131224192154Bh  20131224192154Bh.jpeg  2013-12-24             11
2  050302-20190317145150Bh  20190317145150Bh.jpeg  2019-03-17              1
3  040401-20220618185332Bh  20220618185332Bh.jpeg  2022-06-18             14
4  040401-20211119103130Ch  20211119103130Ch.jpeg  2021-11-19              5


In [66]:
!find /kaggle/input -name "MAGFiLO_1.0_Annotations_kaggle2026_train.json" -type f

/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
/kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-7/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-3/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
/kaggle/input/datasets/nytsoul/solar-filament-s

In [67]:
!find /kaggle/input -iname "*Annotations*kaggle2026*json" -type f

/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
/kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-7/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-3/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
/kaggle/input/datasets/nytsoul/solar-filament-s

In [68]:
!cat /kaggle/working/baseline_v3.yaml

# configs/baseline_v3.yaml
# Full-Image Baseline v3: Region-aware training + Focal/Tversky Loss + Bias Init
dataset:
  train_csv: "outputs/kaggle_train_split.csv"
  val_csv: "outputs/kaggle_val_split.csv"
  image_dir: "MAGFiLO_1.0_Kaggle_2026/train/train_images"
  json_path: "MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
  patch_size: 768

model:
  architecture: "UnetPlusPlus"
  encoder_name: "resnet34"
  encoder_weights: "imagenet"
  in_channels: 1
  classes: 1
  init_bias_prior: 0.01  # Expected foreground prior

loss:
  focal_alpha: 0.25      # Lower alpha downweights background (if alpha is for foreground class)
  focal_gamma: 2.0       # Standard focusing parameter
  tversky_alpha: 0.7     # Weight for false positives
  tversky_beta: 0.3      # Weight for false negatives
  tversky_smooth: 1.0

training:
  batch_size: 4
  learning_rate: 0.0001
  weight_decay: 0.0001
  epochs: 40
  num_workers: 4
  seed: 42
  mixed_precision: true
  gradient_clip_val: 

In [69]:
import os

print("V3 checkpoint:", os.path.exists(
    "/kaggle/working/checkpoints/best_baseline_v3.pth"
))

print("V3 config:", os.path.exists(
    "/kaggle/working/baseline_v3.yaml"
))

print("Validation CSV:", os.path.exists(
    "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv"
))

print("Annotation JSON:", os.path.exists(
    "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
))

V3 checkpoint: True
V3 config: True
Validation CSV: True
Annotation JSON: True


In [76]:
import os
import torch
import yaml

# --------------------------------------------------
# PATHS
# --------------------------------------------------
config_path = "/kaggle/working/baseline_v3.yaml"

val_csv_path = "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv"

json_path = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"

image_dir = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/train_images"

checkpoint_path = "/kaggle/working/checkpoints/best_baseline_v3.pth"

# --------------------------------------------------
# DEVICE
# --------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Checkpoint exists:", os.path.exists(checkpoint_path))

# --------------------------------------------------
# LOAD CONFIG
# --------------------------------------------------
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("Config loaded:", config_path)

# --------------------------------------------------
# LOAD MODEL
# --------------------------------------------------
model = get_baseline_model(config).to(device)

# --------------------------------------------------
# LOAD V3 CHECKPOINT
# --------------------------------------------------
state_dict = torch.load(
    checkpoint_path,
    map_location=device
)

# Handle DataParallel checkpoint
if list(state_dict.keys())[0].startswith("module."):
    print("Detected DataParallel checkpoint. Removing module. prefix...")
    state_dict = {
        k.replace("module.", "", 1): v
        for k, v in state_dict.items()
    }

model.load_state_dict(state_dict, strict=True)

model.eval()

print("✅ V3 checkpoint loaded successfully")
print("Checkpoint:", checkpoint_path)

Device: cuda
Checkpoint exists: True
Config loaded: /kaggle/working/baseline_v3.yaml


NameError: name 'get_baseline_model' is not defined

In [77]:
!grep -R -n "def get_baseline_model" /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/scripts /kaggle/working 2>/dev/null

/kaggle/working/src/models/unet_baseline.py:13:def get_baseline_model(config):
/kaggle/working/.virtual_documents/__notebook_source__.ipynb:790:get_ipython().getoutput("grep -R -n "def get_baseline_model" /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/scripts /kaggle/working 2>/dev/null")


In [78]:
import sys

sys.path.insert(
    0,
    "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/scripts"
)

print("Script path added")

Script path added


In [79]:
from train_baseline_v3 import get_baseline_model

print("✅ get_baseline_model imported")

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


✅ get_baseline_model imported


In [85]:
!grep -n -B 5 -A 25 "def get_baseline_model" \
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/scripts/train_baseline_v3.py

In [86]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Build exact V3 model
model = get_baseline_model(config).to(device)

print("✅ V3 model created")

# Load V3 checkpoint
state_dict = torch.load(
    checkpoint_path,
    map_location=device
)

# Remove DataParallel prefix if present
if list(state_dict.keys())[0].startswith("module."):
    print("Detected DataParallel checkpoint. Stripping module. prefix...")
    state_dict = {
        k.replace("module.", "", 1): v
        for k, v in state_dict.items()
    }

# Strict loading
model.load_state_dict(state_dict, strict=True)
model.eval()

print("✅ V3 checkpoint loaded successfully")
print("Checkpoint:", checkpoint_path)

✅ V3 model created
Detected DataParallel checkpoint. Stripping module. prefix...
✅ V3 checkpoint loaded successfully
Checkpoint: /kaggle/working/checkpoints/best_baseline_v3.pth


In [87]:
import torch

with torch.no_grad():
    dummy = torch.zeros(1, 1, 768, 768).to(device)

    output = model(dummy)

    if isinstance(output, dict):
        output = output["out"]

    prob = torch.sigmoid(output)

print("\n" + "=" * 60)
print("V3 MODEL SANITY CHECK")
print("=" * 60)
print(f"Probability min    : {prob.min().item():.6f}")
print(f"Probability max    : {prob.max().item():.6f}")
print(f"Probability mean   : {prob.mean().item():.6f}")
print(f"Probability median : {prob.median().item():.6f}")
print(f"Foreground > 0.50  : {(prob > 0.50).float().mean().item()*100:.4f}%")
print("=" * 60)


V3 MODEL SANITY CHECK
Probability min    : 0.000005
Probability max    : 0.000797
Probability mean   : 0.000014
Probability median : 0.000014
Foreground > 0.50  : 0.0000%


In [88]:
!grep -R -n "def predict_full_image" \
/kaggle/working \
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation \
2>/dev/null | head -20

/kaggle/working/src/inference/sliding_window.py:5:def predict_full_image(model, image_tensor, patch_size=768, overlap=0.25, device=None):
/kaggle/working/.virtual_documents/__notebook_source__.ipynb:865:get_ipython().getoutput("grep -R -n "def predict_full_image" \")
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/src/inference/sliding_window.py:5:def predict_full_image(model, image_tensor, patch_size=768, overlap=0.25, device=None):


In [89]:
!grep -R -n "predict_full_image" \
/kaggle/working \
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation \
2>/dev/null | head -30

/kaggle/working/src/inference/sliding_window.py:5:def predict_full_image(model, image_tensor, patch_size=768, overlap=0.25, device=None):
/kaggle/working/diagnose_thresholds.py:18:from src.inference.sliding_window import predict_full_image
/kaggle/working/diagnose_thresholds.py:83:            prob_map = predict_full_image(
/kaggle/working/scratch/test_inference.py:13:from src.inference.sliding_window import predict_full_image
/kaggle/working/scratch/test_inference.py:65:            prob_map_v2 = predict_full_image(
/kaggle/working/scratch/test_inference.py:82:            prob_map_pq = predict_full_image(
/kaggle/working/scripts/train_baseline_v3.py:33:from src.inference.sliding_window import predict_full_image
/kaggle/working/scripts/train_baseline_v3.py:136:            prob_map = predict_full_image(model, img_tensor, patch_size=patch_size, overlap=overlap, device=device)
/kaggle/working/scripts/train_baseline_v3.py:189:            prob_map = predict_full_image(
/kaggle/working/scripts

In [90]:
import os
import sys
import torch
import yaml
import numpy as np
import pandas as pd

sys.path.insert(0, "/kaggle/working")
sys.path.insert(0, "/kaggle/working/scripts")

from train_baseline_v3 import get_baseline_model
from src.inference.sliding_window import predict_full_image

# --------------------------------------------------
# PATHS
# --------------------------------------------------

config_path = "/kaggle/working/baseline_v3.yaml"

checkpoint_path = (
    "/kaggle/working/checkpoints/best_baseline_v3.pth"
)

val_csv_path = (
    "/kaggle/input/datasets/nytsoul/"
    "solar-filament-segmentation-9/"
    "solar-filament-segmentation/outputs/"
    "kaggle_val_split.csv"
)

image_dir = (
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026/train/train_images"
)

# --------------------------------------------------
# LOAD CONFIG
# --------------------------------------------------

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Checkpoint:", os.path.exists(checkpoint_path))
print("Validation CSV:", os.path.exists(val_csv_path))
print("Image directory:", os.path.exists(image_dir))

# --------------------------------------------------
# LOAD MODEL
# --------------------------------------------------

model = get_baseline_model(config).to(device)

state_dict = torch.load(
    checkpoint_path,
    map_location=device
)

if list(state_dict.keys())[0].startswith("module."):
    state_dict = {
        k.replace("module.", ""): v
        for k, v in state_dict.items()
    }

model.load_state_dict(state_dict, strict=True)
model.eval()

print("V3 checkpoint loaded successfully")

# --------------------------------------------------
# LOAD ONE VALIDATION IMAGE
# --------------------------------------------------

df = pd.read_csv(val_csv_path)

print("Validation images:", len(df))
print(df.head())


Device: cuda
Checkpoint: True
Validation CSV: True
Image directory: True
V3 checkpoint loaded successfully
Validation images: 142
                  image_id              file_name         day  num_filaments
0  010102-20141027025054Uh  20141027025054Uh.jpeg  2014-10-27             14
1  010201-20131224192154Bh  20131224192154Bh.jpeg  2013-12-24             11
2  050302-20190317145150Bh  20190317145150Bh.jpeg  2019-03-17              1
3  040401-20220618185332Bh  20220618185332Bh.jpeg  2022-06-18             14
4  040401-20211119103130Ch  20211119103130Ch.jpeg  2021-11-19              5


In [91]:
import inspect
from src.data.dataset_fullimg import MAGFiLOFullImageDataset

print(inspect.signature(MAGFiLOFullImageDataset))
print(inspect.getsource(MAGFiLOFullImageDataset.__init__))

(csv_path, img_dir, json_path, transform=None)
    def __init__(self, csv_path, img_dir, json_path, transform=None):
        """
        Args:
            csv_path: Path to CSV with columns [image_id, file_name, ...].
            img_dir: Directory containing the images.
            json_path: Path to the COCO-format annotation JSON.
            transform: Albumentations transform (should be get_inference_augmentation()).
        """
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.parser = MAGFiLOAnnotationParser(json_path)
        self.transform = transform



In [92]:
import src.inference.sliding_window as sw

print("Sliding window implementation:")
print(sw.__file__)

print("\nFunction:")
print(inspect.getsource(sw.predict_full_image))

Sliding window implementation:
/kaggle/working/src/inference/sliding_window.py

Function:
def predict_full_image(model, image_tensor, patch_size=768, overlap=0.25, device=None):
    """
    Run sliding-window inference on a full-resolution image.
    
    Args:
        model: PyTorch model.
        image_tensor: Tensor of shape (1, C, H, W).
        patch_size: Size of the patches (default: 768).
        overlap: Fraction of patch_size to overlap (default: 0.25).
        device: Device to run inference on.
        
    Returns:
        np.ndarray: Probability map of shape (H, W).
    """
    if device is None:
        device = next(model.parameters()).device
        
    model.eval()
    
    _, C, H, W = image_tensor.shape
    stride = int(patch_size * (1 - overlap))
    
    prob_map = torch.zeros((H, W), dtype=torch.float32, device=device)
    count_map = torch.zeros((H, W), dtype=torch.float32, device=device)
    
    # Calculate padded dimensions if necessary
    pad_h = (patch_si

In [94]:
val_dataset = MAGFiLOFullImageDataset(
    csv_path=val_csv_path,
    image_dir=image_dir,
    json_path=json_path,
)

print("Dataset size:", len(val_dataset))

TypeError: MAGFiLOFullImageDataset.__init__() got an unexpected keyword argument 'image_dir'

In [95]:
import inspect
from src.data.dataset_fullimg import MAGFiLOFullImageDataset

print(inspect.signature(MAGFiLOFullImageDataset))
print("\n--- SOURCE ---")
print(inspect.getsource(MAGFiLOFullImageDataset.__init__))

(csv_path, img_dir, json_path, transform=None)

--- SOURCE ---
    def __init__(self, csv_path, img_dir, json_path, transform=None):
        """
        Args:
            csv_path: Path to CSV with columns [image_id, file_name, ...].
            img_dir: Directory containing the images.
            json_path: Path to the COCO-format annotation JSON.
            transform: Albumentations transform (should be get_inference_augmentation()).
        """
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.parser = MAGFiLOAnnotationParser(json_path)
        self.transform = transform



In [96]:
!sed -n '1,100p' /kaggle/working/src/data/dataset_fullimg.py

"""
Full-image validation dataset for 2048x2048 solar images.

Returns full-resolution images and masks for use with sliding-window inference
during validation. Batch size must be 1.
"""
import os
import cv2
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
import sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))
from src.data.annotations import MAGFiLOAnnotationParser
from src.data.masks import create_semantic_mask


class MAGFiLOFullImageDataset(Dataset):
    """
    Dataset that returns full 2048x2048 images with their semantic masks.
    
    For use during validation with sliding-window inference.
    Images are normalized using get_inference_augmentation() externally.
    """

    def __init__(self, csv_path, img_dir, json_path, transform=None):
        """
        Args:
            csv_path: Path to CSV with columns [image_id, file_name, ...].
            img_dir: Directory containing the images.
        

In [97]:
from src.data.dataset_fullimg import MAGFiLOFullImageDataset

val_dataset = MAGFiLOFullImageDataset(
    csv_path=val_csv_path,
    img_dir=image_dir,
    json_path=json_path,
    transform=None
)

print("Dataset size:", len(val_dataset))

Dataset size: 142


In [98]:
config_path = "/kaggle/working/baseline_v3.yaml"

val_csv_path = "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv"

image_dir = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/train_images"

json_path = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"

checkpoint_path = "/kaggle/working/checkpoints/best_baseline_v3.pth"

In [99]:
import os

print("CSV:", os.path.exists(val_csv_path))
print("Images:", os.path.exists(image_dir))
print("JSON:", os.path.exists(json_path))
print("Checkpoint:", os.path.exists(checkpoint_path))

CSV: True
Images: True
JSON: True
Checkpoint: True


In [100]:
val_dataset = MAGFiLOFullImageDataset(
    csv_path=val_csv_path,
    img_dir=image_dir,
    json_path=json_path,
    transform=None
)

print("Dataset size:", len(val_dataset))

Dataset size: 142


In [104]:
from src.data.augmentations import get_inference_augmentation
transform = get_inference_augmentation()

val_dataset = MAGFiLOFullImageDataset(
    csv_path=val_csv_path,
    img_dir=image_dir,
    json_path=json_path,
    transform=transform
)

print("Dataset size:", len(val_dataset))

img_tensor, mask_tensor, filename = val_dataset[0]

print("Filename:", filename)
print("Image shape:", img_tensor.shape)
print("Mask shape:", mask_tensor.shape)
print("Image min:", img_tensor.min().item())
print("Image max:", img_tensor.max().item())
print("Image mean:", img_tensor.mean().item())
print("Mask foreground %:", (mask_tensor > 0).float().mean().item() * 100)

Dataset size: 142
Filename: 20141027025054Uh.jpeg
Image shape: torch.Size([1, 2048, 2048])
Mask shape: torch.Size([1, 2048, 2048])
Image min: -1.0
Image max: 1.0
Image mean: -0.3406679928302765
Mask foreground %: 1.00860595703125


In [105]:
import torch
import numpy as np

model.eval()

img_tensor, mask_tensor, filename = val_dataset[0]

print("=" * 60)
print("V3 SINGLE IMAGE INFERENCE TEST")
print("=" * 60)
print("Filename:", filename)
print("Input shape:", img_tensor.shape)
print("Input min:", img_tensor.min().item())
print("Input max:", img_tensor.max().item())
print("Input mean:", img_tensor.mean().item())

# Add batch dimension
img_tensor = img_tensor.unsqueeze(0).to(device)

with torch.no_grad():
    # Direct patch/full-image model test is NOT used for final metric.
    # Use the exact V3 sliding-window inference.
    prob_map = predict_full_image(
        model,
        img_tensor,
        patch_size=768,
        overlap=0.25,
        device=device
    )

print("\nProbability statistics:")
print("Min   :", float(prob_map.min()))
print("Max   :", float(prob_map.max()))
print("Mean  :", float(prob_map.mean()))
print("Median:", float(np.median(prob_map)))
print("P90   :", float(np.percentile(prob_map, 90)))
print("P95   :", float(np.percentile(prob_map, 95)))
print("P99   :", float(np.percentile(prob_map, 99)))

print("\nThreshold percentages:")
for t in [0.01, 0.02, 0.03, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:
    fg = (prob_map >= t).mean() * 100
    print(f">{t:.2f}: {fg:.4f}%")

V3 SINGLE IMAGE INFERENCE TEST
Filename: 20141027025054Uh.jpeg
Input shape: torch.Size([1, 2048, 2048])
Input min: -1.0
Input max: 1.0
Input mean: -0.3406679928302765

Probability statistics:
Min   : 9.536742027194123e-07
Max   : 1.0
Mean  : 0.005732346326112747
Median: 1.2010335922241211e-05
P90   : 1.6987321941996925e-05
P95   : 2.211332139268052e-05
P99   : 0.00011348724365234375

Threshold percentages:
>0.01: 0.6850%
>0.02: 0.6675%
>0.03: 0.6588%
>0.05: 0.6487%
>0.10: 0.6344%
>0.20: 0.6170%
>0.30: 0.6043%
>0.40: 0.5938%
>0.50: 0.5650%


In [106]:
state_dict = torch.load(
    "/kaggle/working/checkpoints/best_baseline_v3.pth",
    map_location=device
)

if list(state_dict.keys())[0].startswith("module."):
    state_dict = {
        k.replace("module.", "", 1): v
        for k, v in state_dict.items()
    }

model.load_state_dict(state_dict, strict=True)
model.eval()

UnetPlusPlus(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=Tru

In [107]:
import torch
import numpy as np

# Get one validation image
img_tensor, mask_tensor, filename = val_dataset[0]

print("=" * 60)
print("V3 SINGLE-IMAGE INFERENCE CHECK")
print("=" * 60)
print("Filename:", filename)
print("Image:", img_tensor.shape)
print("Mask:", mask_tensor.shape)

# Model inference WITHOUT sliding window first
model.eval()

with torch.no_grad():
    x = img_tensor.unsqueeze(0).to(device)

    logits = model(x)

    probs = torch.sigmoid(logits)

print("\nDirect full-image model output:")
print("Logits min   :", logits.min().item())
print("Logits max   :", logits.max().item())
print("Logits mean  :", logits.mean().item())
print("Prob min     :", probs.min().item())
print("Prob max     :", probs.max().item())
print("Prob mean    :", probs.mean().item())
print("Prob median  :", probs.median().item())
print("FG > 0.50    :", (probs > 0.5).float().mean().item() * 100, "%")

print("\nGround truth:")
print("GT FG %      :", (mask_tensor > 0).float().mean().item() * 100)

print("=" * 60)

V3 SINGLE-IMAGE INFERENCE CHECK
Filename: 20141027025054Uh.jpeg
Image: torch.Size([1, 2048, 2048])
Mask: torch.Size([1, 2048, 2048])

Direct full-image model output:
Logits min   : -13.958013534545898
Logits max   : 15.675106048583984
Logits mean  : -11.176116943359375
Prob min     : 8.671842124385876e-07
Prob max     : 0.9999998807907104
Prob mean    : 0.005635102279484272
Prob median  : 1.1900199751835316e-05
FG > 0.50    : 0.5628347396850586 %

Ground truth:
GT FG %      : 1.00860595703125


In [108]:
# ============================================================
# DIRECT vs SLIDING-WINDOW V3 CHECK
# ============================================================

import numpy as np
import torch

model.eval()

img_tensor, mask_tensor, filename = val_dataset[0]

x = img_tensor.unsqueeze(0).to(device)

with torch.no_grad():

    # -----------------------------
    # 1. DIRECT
    # -----------------------------
    logits_direct = model(x)
    prob_direct = torch.sigmoid(logits_direct).squeeze().cpu().numpy()

    # -----------------------------
    # 2. SLIDING WINDOW
    # -----------------------------
    prob_slide = predict_full_image(
        model,
        x,
        patch_size=768,
        overlap=0.25,
        device=device
    )

print("=" * 60)
print("DIRECT vs SLIDING WINDOW")
print("=" * 60)

print("\nDIRECT:")
print("min     :", prob_direct.min())
print("max     :", prob_direct.max())
print("mean    :", prob_direct.mean())
print("median  :", np.median(prob_direct))
print("FG > .5 :", (prob_direct > 0.5).mean() * 100, "%")

print("\nSLIDING WINDOW:")
print("min     :", prob_slide.min())
print("max     :", prob_slide.max())
print("mean    :", prob_slide.mean())
print("median  :", np.median(prob_slide))
print("FG > .5 :", (prob_slide > 0.5).mean() * 100, "%")

print("\nDIFFERENCE:")
diff = np.abs(prob_direct - prob_slide)

print("Mean abs difference :", diff.mean())
print("Max abs difference  :", diff.max())
print("Pixels > 0.01 diff :", (diff > 0.01).mean() * 100, "%")

print("=" * 60)

DIRECT vs SLIDING WINDOW

DIRECT:
min     : 8.671842e-07
max     : 0.9999999
mean    : 0.005635099
median  : 1.19002e-05
FG > .5 : 0.5628347396850586 %

SLIDING WINDOW:
min     : 9.536742e-07
max     : 1.0
mean    : 0.0057323463
median  : 1.2010336e-05
FG > .5 : 0.5650043487548828 %

DIFFERENCE:
Mean abs difference : 0.0005869601
Max abs difference  : 0.9999868
Pixels > 0.01 diff : 0.14410018920898438 %


In [109]:
!grep -nE "baseline_v2|baseline_v3|checkpoint|config|image_dir|json_path|val_csv|predict_full_image|get_baseline_model|UnetPlusPlus|resnet" /kaggle/working/diagnose_thresholds.py

17:from src.models.unet_baseline import get_baseline_model
18:from src.inference.sliding_window import predict_full_image
28:    config_path = '/kaggle/working/baseline_v3.yaml'
29:    with open(config_path, 'r') as f:
30:        config = yaml.safe_load(f)
32:    image_dir = config['dataset']['image_dir']
33:    if not os.path.isabs(image_dir):
34:        image_dir = os.path.join(base_dir, image_dir)
35:    json_path = config['dataset']['json_path']
36:    if not os.path.isabs(json_path):
37:        json_path = os.path.join(base_dir, json_path)
38:    val_csv_path = '/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv'
44:        csv_path=val_csv_path,
45:        img_dir=image_dir,
46:        json_path=json_path,
51:    model = get_baseline_model(config).to(device)
53:    checkpoint_path = '/kaggle/working/checkpoints/best_baseline_v3.pth'
54:    if not os.path.exists(checkpoint_path):
55:        print(f"Warning: local c

In [110]:
!sed -n '1,120p' /kaggle/working/diagnose_thresholds.py

import os
import sys
import yaml
import torch
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from tqdm import tqdm
import argparse

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from src.data.dataset_fullimg import MAGFiLOFullImageDataset
from src.data.augmentations import get_inference_augmentation
from src.data.annotations import MAGFiLOAnnotationParser
from src.data.masks import create_instance_masks
from src.models.unet_baseline import get_baseline_model
from src.inference.sliding_window import predict_full_image
from src.postprocess.instance import semantic_to_instances
from src.metrics.pq import calculate_pq

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--limit', type=int, default=10, help='Limit number of validation images for smoke testing')
    args = parser.parse_args()

    base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    config_path = '/kaggle/working/

In [115]:
if not os.path.exists(checkpoint_path):
    print(...)
    checkpoint_path = '/kaggle/working/checkpoints/best_baseline_v3.pth'

In [116]:
checkpoint_path = '/kaggle/working/checkpoints/best_baseline_v3.pth'

if not os.path.isfile(checkpoint_path):
    raise FileNotFoundError(
        f"V3 checkpoint not found: {checkpoint_path}"
    )

In [118]:
# ============================================================
# KAGGLE PATHS
# ============================================================

base_dir = "/kaggle/working"

config_path = "/kaggle/working/baseline_v3.yaml"

# Dataset root
dataset_root = (
    "/kaggle/input/datasets/nytsoul/"
    "solar-filament-segmentation-9/"
    "solar-filament-segmentation"
)

image_dir = os.path.join(
    dataset_root,
    "MAGFiLO_1.0_Kaggle_2026",
    "train",
    "train_images"
)

json_path = os.path.join(
    dataset_root,
    "MAGFiLO_1.0_Kaggle_2026",
    "train",
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)

val_csv_path = os.path.join(
    dataset_root,
    "outputs",
    "kaggle_val_split.csv"
)

checkpoint_path = (
    "/kaggle/working/checkpoints/"
    "best_baseline_v3.pth"
)

# ============================================================
# VERIFY PATHS
# ============================================================

print("Config:", os.path.exists(config_path), config_path)
print("Checkpoint:", os.path.exists(checkpoint_path), checkpoint_path)
print("Validation CSV:", os.path.exists(val_csv_path), val_csv_path)
print("Image directory:", os.path.exists(image_dir), image_dir)
print("Annotation JSON:", os.path.exists(json_path), json_path)

if not os.path.exists(json_path):
    raise FileNotFoundError(
        f"Annotation JSON not found:\n{json_path}"
    )

Config: True /kaggle/working/baseline_v3.yaml
Checkpoint: True /kaggle/working/checkpoints/best_baseline_v3.pth
Validation CSV: True /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv
Image directory: True /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/train_images
Annotation JSON: True /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json


In [119]:
val_dataset = MAGFiLOFullImageDataset(
    csv_path=val_csv_path,
    img_dir=image_dir,
    json_path=json_path,
    transform=get_inference_augmentation()
)

print("Dataset size:", len(val_dataset))

Dataset size: 142


In [121]:
!python /kaggle/working/diagnose_thresholds.py --limit 142

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
Traceback (most recent call last):
  File "/kaggle/working/diagnose_thresholds.py", line 181, in <module>
    main()
  File "/kaggle/working/diagnose_thresholds.py", line 43, in main
    val_dataset = MAGFiLOFullImageDataset(
                  ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/src/data/dataset_fullimg.py", line 37, in __init__
    self.parser = MAGFiLOAnnotationParser(json_path)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/src/data/annotations.py", line 7, in __init__
    with open(json_path, 'r') as f:
         ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'


In [122]:
image_dir = config['dataset']['image_dir']
if not os.path.isabs(image_dir):
    image_dir = os.path.join(base_dir, image_dir)

json_path = config['dataset']['json_path']
if not os.path.isabs(json_path):
    json_path = os.path.join(base_dir, json_path)

In [123]:
# ============================================================
# KAGGLE DATASET PATHS
# ============================================================

dataset_root = (
    "/kaggle/input/datasets/nytsoul/"
    "solar-filament-segmentation-9/"
    "solar-filament-segmentation"
)

image_dir = os.path.join(
    dataset_root,
    "MAGFiLO_1.0_Kaggle_2026",
    "train",
    "train_images"
)

json_path = os.path.join(
    dataset_root,
    "MAGFiLO_1.0_Kaggle_2026",
    "train",
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)

val_csv_path = os.path.join(
    dataset_root,
    "outputs",
    "kaggle_val_split.csv"
)

print("Image directory:", image_dir)
print("Annotation JSON:", json_path)
print("Validation CSV:", val_csv_path)

assert os.path.exists(image_dir), f"Image directory missing: {image_dir}"
assert os.path.exists(json_path), f"Annotation JSON missing: {json_path}"
assert os.path.exists(val_csv_path), f"Validation CSV missing: {val_csv_path}"

Image directory: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/train_images
Annotation JSON: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
Validation CSV: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv


In [124]:
checkpoint_path = "/kaggle/working/checkpoints/best_baseline_v3.pth"

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(
        f"V3 checkpoint not found: {checkpoint_path}"
    )

print("Using V3 checkpoint:", checkpoint_path)

Using V3 checkpoint: /kaggle/working/checkpoints/best_baseline_v3.pth


In [125]:
!python /kaggle/working/diagnose_thresholds.py --limit 142

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
Traceback (most recent call last):
  File "/kaggle/working/diagnose_thresholds.py", line 181, in <module>
    main()
  File "/kaggle/working/diagnose_thresholds.py", line 43, in main
    val_dataset = MAGFiLOFullImageDataset(
                  ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/src/data/dataset_fullimg.py", line 37, in __init__
    self.parser = MAGFiLOAnnotationParser(json_path)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/src/data/annotations.py", line 7, in __init__
    with open(json_path, 'r') as f:
         ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'


In [126]:
!sed -n '25,50p' /kaggle/working/diagnose_thresholds.py


    args = parser.parse_args()

    base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    config_path = '/kaggle/working/baseline_v3.yaml'
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    image_dir = config['dataset']['image_dir']
    if not os.path.isabs(image_dir):
        image_dir = os.path.join(base_dir, image_dir)
    json_path = config['dataset']['json_path']
    if not os.path.isabs(json_path):
        json_path = os.path.join(base_dir, json_path)
    val_csv_path = '/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv'
    
    outputs_dir = os.path.join(base_dir, 'outputs')
    os.makedirs(outputs_dir, exist_ok=True)
    
    val_dataset = MAGFiLOFullImageDataset(
        csv_path=val_csv_path,
        img_dir=image_dir,
        json_path=json_path,
        transform=get_inference_augmentation()
    )
    
    device = torch.device('cuda' if torch.cuda.is_availab

In [130]:
image_dir = config['dataset']['image_dir']
if not os.path.isabs(image_dir):
    image_dir = os.path.join(base_dir, image_dir)

json_path = config['dataset']['json_path']
if not os.path.isabs(json_path):
    json_path = os.path.join(base_dir, json_path)

val_csv_path = '/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv'

In [132]:
# ============================================================
# KAGGLE DATASET PATHS
# ============================================================

dataset_root = (
    "/kaggle/input/datasets/nytsoul/"
    "solar-filament-segmentation-9/"
    "solar-filament-segmentation"
)

image_dir = os.path.join(
    dataset_root,
    "MAGFiLO_1.0_Kaggle_2026",
    "train",
    "train_images"
)

json_path = os.path.join(
    dataset_root,
    "MAGFiLO_1.0_Kaggle_2026",
    "train",
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)

val_csv_path = os.path.join(
    dataset_root,
    "outputs",
    "kaggle_val_split.csv"
)

print("Image directory:", image_dir)
print("Annotation JSON:", json_path)
print("Validation CSV:", val_csv_path)

assert os.path.exists(image_dir), f"Missing image directory: {image_dir}"
assert os.path.exists(json_path), f"Missing annotation JSON: {json_path}"
assert os.path.exists(val_csv_path), f"Missing validation CSV: {val_csv_path}"

Image directory: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/train_images
Annotation JSON: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
Validation CSV: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv


In [134]:
from pathlib import Path

path = Path("/kaggle/working/diagnose_thresholds.py")
text = path.read_text()

start = text.index("    image_dir = config['dataset']['image_dir']")
end = text.index("    outputs_dir =", start)

new_block = '''    # ============================================================
    # KAGGLE DATASET PATHS
    # ============================================================

    dataset_root = "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation"

    image_dir = os.path.join(
        dataset_root,
        "MAGFiLO_1.0_Kaggle_2026",
        "train",
        "train_images"
    )

    json_path = os.path.join(
        dataset_root,
        "MAGFiLO_1.0_Kaggle_2026",
        "train",
        "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    )

    val_csv_path = os.path.join(
        dataset_root,
        "outputs",
        "kaggle_val_split.csv"
    )

    print("Image directory:", image_dir)
    print("Annotation JSON:", json_path)
    print("Validation CSV:", val_csv_path)

    assert os.path.exists(image_dir), f"Image directory missing: {image_dir}"
    assert os.path.exists(json_path), f"Annotation JSON missing: {json_path}"
    assert os.path.exists(val_csv_path), f"Validation CSV missing: {val_csv_path}"

'''

text = text[:start] + new_block + text[end:]
path.write_text(text)

print("✅ diagnose_thresholds.py UPDATED")

✅ diagnose_thresholds.py UPDATED


In [135]:
!sed -n '25,75p' /kaggle/working/diagnose_thresholds.py

    args = parser.parse_args()

    base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    config_path = '/kaggle/working/baseline_v3.yaml'
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    # ============================================================
    # KAGGLE DATASET PATHS
    # ============================================================

    dataset_root = "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation"

    image_dir = os.path.join(
        dataset_root,
        "MAGFiLO_1.0_Kaggle_2026",
        "train",
        "train_images"
    )

    json_path = os.path.join(
        dataset_root,
        "MAGFiLO_1.0_Kaggle_2026",
        "train",
        "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    )

    val_csv_path = os.path.join(
        dataset_root,
        "outputs",
        "kaggle_val_split.csv"
    )

    print("Image directory:", image_dir)
    print("Annotation JSON:", json_path

In [136]:
!grep -n "json_path" /kaggle/working/diagnose_thresholds.py

45:    json_path = os.path.join(
59:    print("Annotation JSON:", json_path)
63:    assert os.path.exists(json_path), f"Annotation JSON missing: {json_path}"
72:        json_path=json_path,
90:    parser_ann = MAGFiLOAnnotationParser(json_path)


In [137]:
!ls -lh /kaggle/working/diagnose_thresholds.py

-rw-r--r-- 1 root root 7.6K Sep 22 19:28 /kaggle/working/diagnose_thresholds.py


In [138]:
!grep -n "dataset_root\|json_path =\|MAGFiLO_1.0_Annotations" /kaggle/working/diagnose_thresholds.py

36:    dataset_root = "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation"
39:        dataset_root,
45:    json_path = os.path.join(
46:        dataset_root,
49:        "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
53:        dataset_root,


In [142]:
dataset_root = "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation"

image_dir = os.path.join(
    dataset_root,
    "MAGFiLO_1.0_Kaggle_2026",
    "train",
    "train_images"
)

json_path = os.path.join(
    dataset_root,
    "MAGFiLO_1.0_Kaggle_2026",
    "train",
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)

val_csv_path = os.path.join(
    dataset_root,
    "outputs",
    "kaggle_val_split.csv"
)

In [143]:
import os

dataset_root = "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation"

paths = {
    "ROOT": dataset_root,
    "IMAGES": os.path.join(
        dataset_root,
        "MAGFiLO_1.0_Kaggle_2026",
        "train",
        "train_images"
    ),
    "JSON": os.path.join(
        dataset_root,
        "MAGFiLO_1.0_Kaggle_2026",
        "train",
        "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    ),
    "CSV": os.path.join(
        dataset_root,
        "outputs",
        "kaggle_val_split.csv"
    ),
    "CHECKPOINT": "/kaggle/working/checkpoints/best_baseline_v3.pth"
}

for name, path in paths.items():
    print(f"{name}:")
    print(path)
    print("EXISTS:", os.path.exists(path))
    print()

ROOT:
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation
EXISTS: True

IMAGES:
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/train_images
EXISTS: True

JSON:
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
EXISTS: True

CSV:
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv
EXISTS: True

CHECKPOINT:
/kaggle/working/checkpoints/best_baseline_v3.pth
EXISTS: True



In [144]:
!find /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation \
-name "*.json" -print

/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/full_image_baseline_v2/training_history_v2.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/full_image_baseline_v2/final_metrics_v2.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/final_metrics.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_integration_report.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/diagnostic_consistency/consistency_summary.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/threshold_diagnostic.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/training_signal_diagnostic.json
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filam

In [145]:
!grep -n "dataset_root\|json_path\|MAGFiLO_1.0_Annotations" /kaggle/working/diagnose_thresholds.py

36:    dataset_root = "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation"
39:        dataset_root,
45:    json_path = os.path.join(
46:        dataset_root,
49:        "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
53:        dataset_root,
59:    print("Annotation JSON:", json_path)
63:    assert os.path.exists(json_path), f"Annotation JSON missing: {json_path}"
72:        json_path=json_path,
90:    parser_ann = MAGFiLOAnnotationParser(json_path)


In [146]:
!python /kaggle/working/diagnose_thresholds.py --limit 5

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
Image directory: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/train_images
Annotation JSON: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
Validation CSV: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv
Processing images: 100%|██████████████████████████| 5/5 [00:14<00:00,  2.92s/it]

--- Threshold Diagnostic Summary ---
 threshold  mean_dice  mean_iou  mean_fg_pct  mean_pred_cc  mean_gt_cc  mean_tp  mean_fp  mean_fn
      0.10   0.481474  0.341574     0.449491          14.6         7.6      4.2     10.4      3.4
      0.

In [147]:
!python /kaggle/working/diagnose_thresholds.py --limit 142

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
Image directory: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/train_images
Annotation JSON: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
Validation CSV: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv
Processing images: 100%|██████████████████████| 142/142 [06:27<00:00,  2.73s/it]

--- Threshold Diagnostic Summary ---
 threshold  mean_dice  mean_iou  mean_fg_pct  mean_pred_cc  mean_gt_cc  mean_tp  mean_fp  mean_fn
      0.10   0.577912  0.427274     0.359297     12.091549    7.105634 3.992958 8.098592 3.112676
      0.

In [148]:
!grep -R -n "def semantic_to_instances" /kaggle/working/src

/kaggle/working/src/postprocess/instance.py:4:def semantic_to_instances(prob_map: np.ndarray, prob_threshold: float = 0.5, min_area: int = 100) -> np.ndarray:


In [149]:
!grep -R -n "def semantic_to_instances" /kaggle/working/src

/kaggle/working/src/postprocess/instance.py:4:def semantic_to_instances(prob_map: np.ndarray, prob_threshold: float = 0.5, min_area: int = 100) -> np.ndarray:


In [150]:
!sed -n '1,220p' /kaggle/working/src/postprocess/instance.py

import numpy as np
import cv2

def semantic_to_instances(prob_map: np.ndarray, prob_threshold: float = 0.5, min_area: int = 100) -> np.ndarray:
    """
    Convert a semantic probability map to an instance mask using connected components.
    
    Args:
        prob_map: 2D numpy array with probabilities [0, 1].
        prob_threshold: Threshold to binarize the probability map.
        min_area: Minimum area (pixels) for a component to be kept.
        
    Returns:
        instance_mask: 2D numpy array where 0 is background and 1..N are unique instance IDs.
    """
    binary_mask = (prob_map >= prob_threshold).astype(np.uint8)
    
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    
    instance_mask = np.zeros_like(labels, dtype=np.int32)
    current_instance_id = 1
    
    # stats[0] is the background
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_area:
            instan

In [151]:
# ============================================================
# V3 POST-PROCESSING GRID SEARCH
# Tests threshold + min_area using the existing V3 checkpoint
# ============================================================

import os
import sys
import yaml
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.append("/kaggle/working")

from src.data.dataset_fullimg import MAGFiLOFullImageDataset
from src.data.augmentations import get_inference_augmentation
from src.data.annotations import MAGFiLOAnnotationParser
from src.data.masks import create_instance_masks
from src.models.unet_baseline import get_baseline_model
from src.inference.sliding_window import predict_full_image
from src.postprocess.instance import semantic_to_instances
from src.metrics.pq import calculate_pq


# ============================================================
# PATHS
# ============================================================

dataset_root = (
    "/kaggle/input/datasets/nytsoul/"
    "solar-filament-segmentation-9/"
    "solar-filament-segmentation"
)

image_dir = os.path.join(
    dataset_root,
    "MAGFiLO_1.0_Kaggle_2026",
    "train",
    "train_images"
)

json_path = os.path.join(
    dataset_root,
    "MAGFiLO_1.0_Kaggle_2026",
    "train",
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)

val_csv_path = os.path.join(
    dataset_root,
    "outputs",
    "kaggle_val_split.csv"
)

checkpoint_path = (
    "/kaggle/working/checkpoints/"
    "best_baseline_v3.pth"
)

config_path = "/kaggle/working/baseline_v3.yaml"


# ============================================================
# CONFIG
# ============================================================

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Checkpoint exists:", os.path.exists(checkpoint_path))
print("Validation CSV exists:", os.path.exists(val_csv_path))
print("JSON exists:", os.path.exists(json_path))


# ============================================================
# DATASET
# ============================================================

val_dataset = MAGFiLOFullImageDataset(
    csv_path=val_csv_path,
    img_dir=image_dir,
    json_path=json_path,
    transform=get_inference_augmentation()
)

parser_ann = MAGFiLOAnnotationParser(json_path)

print("Validation images:", len(val_dataset))


# ============================================================
# MODEL
# ============================================================

model = get_baseline_model(config).to(device)

state_dict = torch.load(
    checkpoint_path,
    map_location=device
)

if list(state_dict.keys())[0].startswith("module."):
    state_dict = {
        k.replace("module.", ""): v
        for k, v in state_dict.items()
    }

model.load_state_dict(state_dict, strict=True)
model.eval()

print("V3 checkpoint loaded successfully.")


# ============================================================
# SETTINGS TO TEST
# ============================================================

thresholds = [
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55
]

min_areas = [
    50,
    100,
    150,
    200,
    300,
    500
]


# ============================================================
# FIRST: GENERATE PROBABILITY MAPS ONLY ONCE
# ============================================================

prob_maps = []
gt_instances_list = []

print("\nGenerating probability maps...")

for i in tqdm(
    range(len(val_dataset)),
    desc="Inference"
):

    img_tensor, mask_tensor, filename = val_dataset[i]

    img_tensor = img_tensor.unsqueeze(0).to(device)

    gt_semantic = (
        mask_tensor
        .squeeze()
        .numpy()
        .astype(np.uint8)
    )

    img_id = parser_ann.filename_to_img_id.get(filename)

    anns = parser_ann.get_annotations_for_image(img_id)

    gt_instances = create_instance_masks(
        anns,
        height=gt_semantic.shape[0],
        width=gt_semantic.shape[1]
    )

    with torch.no_grad():

        prob_map = predict_full_image(
            model,
            img_tensor,
            patch_size=config["dataset"].get(
                "patch_size",
                768
            ),
            overlap=config.get(
                "validation",
                {}
            ).get(
                "overlap",
                0.25
            ),
            device=device
        )

    prob_maps.append(prob_map)
    gt_instances_list.append(gt_instances)


print("Probability maps generated:", len(prob_maps))


# ============================================================
# GRID SEARCH
# ============================================================

results = []

total = len(thresholds) * len(min_areas)

print("\n")
print("=" * 70)
print("V3 POST-PROCESSING GRID SEARCH")
print("=" * 70)
print("Combinations:", total)


for threshold in thresholds:

    for min_area in min_areas:

        pq_values = []
        sq_values = []
        rq_values = []

        tp_total = 0
        fp_total = 0
        fn_total = 0

        pred_cc_total = 0
        gt_cc_total = 0

        for prob_map, gt_instances in zip(
            prob_maps,
            gt_instances_list
        ):

            pred_instances = semantic_to_instances(
                prob_map,
                prob_threshold=threshold,
                min_area=min_area
            )

            num_pred = int(
                pred_instances.max()
            )

            num_gt = int(
                gt_instances.max()
            )

            pred_cc_total += num_pred
            gt_cc_total += num_gt

            metrics = calculate_pq(
                pred_instances,
                gt_instances
            )

            pq_values.append(
                metrics["pq"]
            )

            sq_values.append(
                metrics["sq"]
            )

            rq_values.append(
                metrics["rq"]
            )

            tp_total += metrics["tp"]
            fp_total += metrics["fp"]
            fn_total += metrics["fn"]

        result = {
            "threshold": threshold,
            "min_area": min_area,
            "PQ": np.mean(pq_values),
            "SQ": np.mean(sq_values),
            "RQ": np.mean(rq_values),
            "TP": tp_total,
            "FP": fp_total,
            "FN": fn_total,
            "pred_instances_per_image":
                pred_cc_total / len(val_dataset),
            "gt_instances_per_image":
                gt_cc_total / len(val_dataset)
        }

        results.append(result)

        print(
            f"threshold={threshold:.2f} "
            f"min_area={min_area:3d} | "
            f"PQ={result['PQ']:.4f} | "
            f"SQ={result['SQ']:.4f} | "
            f"RQ={result['RQ']:.4f} | "
            f"TP={tp_total:4d} "
            f"FP={fp_total:4d} "
            f"FN={fn_total:4d} | "
            f"Pred={result['pred_instances_per_image']:.2f}"
        )


# ============================================================
# RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "PQ",
    ascending=False
).reset_index(drop=True)

print("\n")
print("=" * 70)
print("TOP POST-PROCESSING CONFIGURATIONS")
print("=" * 70)

display(
    results_df.head(15)
)


# ============================================================
# SAVE
# ============================================================

output_path = (
    "/kaggle/working/"
    "v3_postprocess_grid_search.csv"
)

results_df.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)

Device: cuda
Checkpoint exists: True
Validation CSV exists: True
JSON exists: True
Validation images: 142
V3 checkpoint loaded successfully.

Generating probability maps...


Inference: 100%|██████████| 142/142 [01:59<00:00,  1.19it/s]


Probability maps generated: 142


V3 POST-PROCESSING GRID SEARCH
Combinations: 36
threshold=0.30 min_area= 50 | PQ=0.2640 | SQ=0.6300 | RQ=0.3927 | TP= 551 FP=1264 FN= 458 | Pred=12.78
threshold=0.30 min_area=100 | PQ=0.2903 | SQ=0.6300 | RQ=0.4313 | TP= 551 FP=1039 FN= 458 | Pred=11.20
threshold=0.30 min_area=150 | PQ=0.3049 | SQ=0.6300 | RQ=0.4529 | TP= 551 FP= 912 FN= 458 | Pred=10.30
threshold=0.30 min_area=200 | PQ=0.3125 | SQ=0.6293 | RQ=0.4645 | TP= 548 FP= 826 FN= 461 | Pred=9.68
threshold=0.30 min_area=300 | PQ=0.3281 | SQ=0.6290 | RQ=0.4873 | TP= 538 FP= 666 FN= 471 | Pred=8.48
threshold=0.30 min_area=500 | PQ=0.3293 | SQ=0.6173 | RQ=0.4893 | TP= 499 FP= 483 FN= 510 | Pred=6.92
threshold=0.35 min_area= 50 | PQ=0.2659 | SQ=0.6259 | RQ=0.3951 | TP= 546 FP=1249 FN= 463 | Pred=12.64
threshold=0.35 min_area=100 | PQ=0.2896 | SQ=0.6259 | RQ=0.4299 | TP= 546 FP=1024 FN= 463 | Pred=11.06
threshold=0.35 min_area=150 | PQ=0.3043 | SQ=0.6259 | RQ=0.4515 | TP= 546 FP= 902 FN= 463 | Pred=

,threshold,min_area,PQ,SQ,RQ,TP,FP,FN,pred_instances_per_image,gt_instances_per_image
0,0.30,500,0.329315,0.617297,0.489307,499,483,510,6.915493,7.105634
1,0.35,500,0.328803,0.616680,0.488684,495,477,514,6.845070,7.105634
2,0.30,300,0.328071,0.628964,0.487340,538,666,471,8.478873,7.105634
3,0.40,500,0.326854,0.617448,0.485277,487,470,522,6.739437,7.105634
4,0.40,300,0.326660,0.624881,0.484819,529,644,480,8.260563,7.105634
5,0.35,300,0.325616,0.624997,0.483162,532,659,477,8.387324,7.105634
6,0.45,300,0.324629,0.621332,0.481156,522,634,487,8.140845,7.105634
7,0.45,500,0.323762,0.613866,0.479747,482,469,527,6.697183,7.105634
8,0.40,200,0.313348,0.625126,0.465411,540,795,469,9.401408,7.105634
9,0.30,200,0.312544,0.629292,0.464466,548,826,461,9.676056,7.105634



Saved: /kaggle/working/v3_postprocess_grid_search.csv


In [5]:
thresholds = [
    0.25, 0.275, 0.30, 0.325,
    0.35, 0.375, 0.40, 0.425
]

min_areas = [
    250, 300, 350, 400,
    450, 500, 550, 600
]

In [14]:
!find /kaggle/working -type f -name "*.py" | grep -Ei "grid|postprocess|pq"

In [17]:
!grep -R -n "V3 POST-PROCESSING GRID SEARCH" /kaggle/working --include="*.py"

In [28]:
!find /kaggle/input -type f -name "train_baseline_v3.py" -print

/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/scripts/train_baseline_v3.py


In [29]:
!find /kaggle/working /kaggle/input -type f \
  \( -name "train*.py" -o -name "*baseline*.py" \) -print

/kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation/scripts/train_baseline.py
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation/scripts/evaluate_baseline.py
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation/src/models/unet_baseline.py
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-8/solar-filament-segmentation/scripts/train_baseline_v2.py
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-8/solar-filament-segmentation/scripts/train_baseline.py
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-8/solar-filament-segmentation/scripts/evaluate_baseline.py
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-8/solar-filament-segmentation/vendor/segmentation_models_pytorch/utils/train.py
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-8/solar-filament-segmentation/src/models/unet_baseline.py
/kaggle/input/datasets/nytsoul/solar-filame

In [30]:
!cp /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/scripts/train_baseline_v3.py \
/kaggle/working/train_baseline_v3.py

!ls -lh /kaggle/working/train_baseline_v3.py

-rw-r--r-- 1 root root 23K Sep 22 23:50 /kaggle/working/train_baseline_v3.py


In [31]:
!grep -nEi "epoch|epochs|resume|checkpoint|load_state_dict|start_epoch|best_baseline_v3" \
/kaggle/working/train_baseline_v3.py

7:- Best checkpoint saved by full-image validation Dice
79:    epochs = [h['epoch'] for h in history]
88:    axes[0, 0].plot(epochs, train_loss, 'o-', label='Train Loss', color='blue')
90:    axes[0, 0].set_xlabel('Epoch')
94:    axes[0, 1].plot(epochs, val_dice, 'o-', label='Val Dice (Full Image)', color='green')
96:    axes[0, 1].set_xlabel('Epoch')
100:    axes[1, 0].plot(epochs, val_iou, 'o-', label='Val IoU (Full Image)', color='orange')
102:    axes[1, 0].set_xlabel('Epoch')
106:    axes[1, 1].plot(epochs, pct_fil, 'o-', label='% Patches with Filament', color='purple')
107:    axes[1, 1].plot(epochs, pct_bg, 's--', label='% Background/Limb/Disk Patches', color='gray')
109:    axes[1, 1].set_xlabel('Epoch')
219:    parser.add_argument('--epochs', type=int, default=None)
242:    checkpoints_dir = os.path.join(out_dir_base, 'checkpoints')
244:    os.makedirs(checkpoints_dir, exist_ok=True)
314:    logger.info(f"Training patches per epoch: {len(train_dataset)} ({len(train_dataset.df)

In [33]:
!grep -nEi "epoch|epochs|checkpoint|resume|scheduler" \
/kaggle/working/baseline_v3.yaml

grep: /kaggle/working/baseline_v3.yaml: No such file or directory


In [35]:
!find /kaggle/input /kaggle/working -type f \( -name "baseline_v3.yaml" -o -name "*.yaml" -o -name "*.yml" \) 2>/dev/null | head -30

/kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation/configs/baseline.yaml
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-8/solar-filament-segmentation/outputs/full_image_baseline_v2/config_v2.yaml
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-8/solar-filament-segmentation/configs/baseline.yaml
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-8/solar-filament-segmentation/configs/baseline_v2.yaml
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-7/solar-filament-segmentation/outputs/full_image_baseline_v2/config_v2.yaml
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-7/solar-filament-segmentation/configs/baseline.yaml
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-7/solar-filament-segmentation/configs/baseline_v2.yaml
/kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation/configs/baseline.yaml
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5

In [36]:
!sed -n '330,530p' /kaggle/working/train_baseline_v3.py

    logger.info(f"Initializing final segmentation_head bias to {bias_val:.4f} for prior {prior}")
    with torch.no_grad():
        model.segmentation_head[0].bias.fill_(bias_val)
        
    # Check initial probability on a dummy tensor
    dummy = torch.zeros(1, config['model'].get('in_channels', 1), 256, 256)
    initial_prob = torch.sigmoid(model(dummy)).mean().item()
    logger.info(f"Initial model mean probability (on zero input): {initial_prob:.4f}")

    multi_gpu = False
    if device.type == 'cuda' and config['training'].get('use_multi_gpu', False) and torch.cuda.device_count() > 1:
        logger.info(f"Using {torch.cuda.device_count()} GPUs for DataParallel")
        model = torch.nn.DataParallel(model)
        multi_gpu = True

    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['training']['learning_rate'],
        weight_decay=config['training'].get('weight_decay', 1e-4)
    )
    epochs = args.epochs if args.ep

In [37]:
!find /kaggle/working -maxdepth 2 -type f -name "*v3*" -o -name "*yaml*" | sort

/kaggle/working/train_baseline_v3.py


In [41]:
!find /kaggle/working -maxdepth 2 -type f \( -name "*v3*" -o -name "*.yaml" \) | sort

/kaggle/working/train_baseline_v3.py


In [42]:
!grep -nE "epochs|learning_rate|weight_decay|scheduler|patch_size|batch_size" \
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/configs/baseline_v3.yaml

8:  patch_size: 768
26:  batch_size: 4
27:  learning_rate: 0.0001
28:  weight_decay: 0.0001
29:  epochs: 40


In [43]:
!grep -nE "epochs|learning_rate|weight_decay|scheduler|patch_size|batch_size" \
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/full_image_baseline_v3/config_v3.yaml

16:  patch_size: 768
39:  batch_size: 4
40:  epochs: 40
42:  learning_rate: 0.0001
48:  weight_decay: 0.0001


In [46]:
!cp /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/configs/baseline_v3.yaml \
/kaggle/working/baseline_v3.yaml

In [48]:
!sed -i 's/^  epochs: 40$/  epochs: 30/' \
/kaggle/working/baseline_v3.yaml

In [49]:
!grep -n "epochs" /kaggle/working/baseline_v3.yaml

29:  epochs: 30


In [51]:
!cp -r /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/src /kaggle/working/

In [53]:
!cp -r /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/vendor /kaggle/working/ 2>/dev/null || true

In [54]:
!ls /kaggle/working/src

data  inference  __init__.py  losses  metrics  models  postprocess


In [55]:
!ls -l /kaggle/working/src/data/

total 44
-rw-r--r-- 1 root root  1474 Sep 22 23:59 annotations.py
-rw-r--r-- 1 root root  2868 Sep 22 23:59 augmentations.py
-rw-r--r-- 1 root root  2604 Sep 22 23:59 dataset_fullimg.py
-rw-r--r-- 1 root root 10651 Sep 22 23:59 dataset.py
-rw-r--r-- 1 root root     7 Sep 22 23:59 __init__.py
-rw-r--r-- 1 root root  2573 Sep 22 23:59 masks.py
-rw-r--r-- 1 root root  5917 Sep 22 23:59 solar_disk.py
-rw-r--r-- 1 root root  3706 Sep 22 23:59 split.py


In [56]:
import sys
sys.path.insert(0, "/kaggle/working")

from src.data.dataset import MAGFiLOPatchDataset

print("src import OK")

src import OK


In [57]:
!python /kaggle/working/train_baseline_v3.py --epochs 30

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
Traceback (most recent call last):
  File "/kaggle/working/train_baseline_v3.py", line 571, in <module>
    main()
  File "/kaggle/working/train_baseline_v3.py", line 230, in main
    with open(config_path, 'r') as f:
         ^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/configs/baseline_v3.yaml'


In [58]:
!grep -n "config_path" /kaggle/working/train_baseline_v3.py

226:    config_path = args.config
227:    if not os.path.isabs(config_path):
228:        config_path = os.path.join(base_dir, config_path)
230:    with open(config_path, 'r') as f:


In [61]:
!sed -i "s|config_path = .*|config_path = '/kaggle/working/baseline_v3.yaml'|" /kaggle/working/train_baseline_v3.py

In [62]:
!grep -n "config_path" /kaggle/working/train_baseline_v3.py

226:    config_path = '/kaggle/working/baseline_v3.yaml'
227:    if not os.path.isabs(config_path):
228:        config_path = '/kaggle/working/baseline_v3.yaml'
230:    with open(config_path, 'r') as f:


In [63]:
!grep -n "epochs:" /kaggle/working/baseline_v3.yaml

29:  epochs: 30


In [64]:
!sed -i '226,229c\    config_path = "/kaggle/working/baseline_v3.yaml"\n    with open(config_path, "r") as f:\n        config = yaml.safe_load(f)' /kaggle/working/train_baseline_v3.py

In [65]:
!sed -n '220,235p' /kaggle/working/train_baseline_v3.py

    parser.add_argument('--out_dir', type=str, default=None)
    parser.add_argument('--val_limit', type=int, default=None,
                        help='Limit number of validation images for smoke testing')
    args = parser.parse_args()

    base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    config_path = "/kaggle/working/baseline_v3.yaml"
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    in_kaggle = configure_kaggle_paths(config, base_dir)

    # Determine outputs path
    if in_kaggle and not args.out_dir:


In [69]:
!sed -n '205,250p' /kaggle/working/train_baseline_v3.py

            iou = intersection / (union + 1e-7) if union > 0 else 0.0

            all_dice.append(dice)
            all_iou.append(iou)

    mean_dice = float(np.mean(all_dice))
    mean_iou = float(np.mean(all_iou))
    return mean_dice, mean_iou


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', type=str, default='configs/baseline_v3.yaml')
    parser.add_argument('--limit_batches', type=int, default=None)
    parser.add_argument('--epochs', type=int, default=None)
    parser.add_argument('--out_dir', type=str, default=None)
    parser.add_argument('--val_limit', type=int, default=None,
                        help='Limit number of validation images for smoke testing')
    args = parser.parse_args()

    base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    config_path = "/kaggle/working/baseline_v3.yaml"
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    with open(config_path, 'r') as f:
        co

In [70]:
from pathlib import Path

path = Path("/kaggle/working/train_baseline_v3.py")
text = path.read_text()

start = text.index("    args = parser.parse_args()")
end = text.index("    # Determine outputs path", start)

replacement = '''    args = parser.parse_args()

    # ============================================================
    # KAGGLE PATH CONFIGURATION
    # ============================================================

    base_dir = "/kaggle/working"

    # V3 configuration
    config_path = "/kaggle/working/baseline_v3.yaml"

    if not os.path.exists(config_path):
        raise FileNotFoundError(
            f"V3 config not found: {config_path}"
        )

    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    print("=" * 60)
    print("KAGGLE V3 TRAINING")
    print("=" * 60)
    print("Config:", config_path)
    print("Epochs:", config["training"]["epochs"])
    print("=" * 60)

    in_kaggle = True

    if args.out_dir:
        out_dir_base = args.out_dir
    else:
        out_dir_base = "/kaggle/working"

    os.makedirs(out_dir_base, exist_ok=True)

'''

text = text[:start] + replacement + text[end:]

path.write_text(text)

print("Updated train_baseline_v3.py")

Updated train_baseline_v3.py


In [71]:
!sed -n '215,260p' /kaggle/working/train_baseline_v3.py

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', type=str, default='configs/baseline_v3.yaml')
    parser.add_argument('--limit_batches', type=int, default=None)
    parser.add_argument('--epochs', type=int, default=None)
    parser.add_argument('--out_dir', type=str, default=None)
    parser.add_argument('--val_limit', type=int, default=None,
                        help='Limit number of validation images for smoke testing')
    args = parser.parse_args()

    # ============================================================
    # KAGGLE PATH CONFIGURATION
    # ============================================================

    base_dir = "/kaggle/working"

    # V3 configuration
    config_path = "/kaggle/working/baseline_v3.yaml"

    if not os.path.exists(config_path):
        raise FileNotFoundError(
            f"V3 config not found: {config_path}"
        )

    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    print("=" 

In [72]:
!grep -n "epochs:" /kaggle/working/baseline_v3.yaml

29:  epochs: 30


In [73]:
!python /kaggle/working/train_baseline_v3.py --help

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
usage: train_baseline_v3.py [-h] [--config CONFIG]
                            [--limit_batches LIMIT_BATCHES] [--epochs EPOCHS]
                            [--out_dir OUT_DIR] [--val_limit VAL_LIMIT]

options:
  -h, --help            show this help message and exit
  --config CONFIG
  --limit_batches LIMIT_BATCHES
  --epochs EPOCHS
  --out_dir OUT_DIR
  --val_limit VAL_LIMIT
                        Limit number of validation images for smoke testing


In [74]:
!python /kaggle/working/train_baseline_v3.py \
    --config /kaggle/working/baseline_v3.yaml \
    --epochs 30 \
    --out_dir /kaggle/working

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
KAGGLE V3 TRAINING
Config: /kaggle/working/baseline_v3.yaml
Epochs: 30
2026-09-23 00:11:57,677 [INFO] Using device: cuda
Traceback (most recent call last):
  File "/kaggle/working/train_baseline_v3.py", line 594, in <module>
    main()
  File "/kaggle/working/train_baseline_v3.py", line 312, in main
    train_dataset = MAGFiLOPatchDataset(
                    ^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/src/data/dataset.py", line 83, in __init__
    self.df = pd.read_csv(csv_path)
              ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py",

In [75]:
!find /kaggle/input /kaggle/working -type f -name "kaggle_train_split.csv" 2>/dev/null

/kaggle/input/datasets/nytsoul/solar-filament-segmentation-4/solar-filament-segmentation/outputs/kaggle_train_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-8/solar-filament-segmentation/outputs/kaggle_train_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-8/solar-filament-segmentation/outputs/full_image_baseline_v2/kaggle_train_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-7/solar-filament-segmentation/outputs/kaggle_train_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-7/solar-filament-segmentation/outputs/full_image_baseline_v2/kaggle_train_split.csv
/kaggle/input/datasets/nytsoul/solor-filament-segmentation/solar-filament-segmentation/outputs/kaggle_train_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-5/solar-filament-segmentation/outputs/kaggle_train_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-6/solar-filament-segmentation/outputs/kaggle_train_split.

In [78]:
!find /kaggle/input /kaggle/working -type f -name "kaggle_val_split.csv" 2>/dev/null!find /kaggle/input /kaggle/working -type f -name "kaggle_val_split.csv" 2>/dev/null

In [79]:
!mkdir -p /kaggle/working/outputs

!cp /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_train_split.csv \
    /kaggle/working/outputs/kaggle_train_split.csv

In [80]:
!ls -lh /kaggle/working/outputs/kaggle_train_split.csv

-rw-r--r-- 1 root root 34K Sep 23 00:13 /kaggle/working/outputs/kaggle_train_split.csv


In [81]:
import os

train_csv = "/kaggle/working/outputs/kaggle_train_split.csv"

print("Train CSV:", os.path.exists(train_csv))
print("Train size:", os.path.getsize(train_csv) if os.path.exists(train_csv) else "MISSING")

Train CSV: True
Train size: 34214


In [82]:
!grep -nE "train_csv|val_csv|image_dir|json_path|epochs" \
/kaggle/working/baseline_v3.yaml

4:  train_csv: "outputs/kaggle_train_split.csv"
5:  val_csv: "outputs/kaggle_val_split.csv"
6:  image_dir: "MAGFiLO_1.0_Kaggle_2026/train/train_images"
7:  json_path: "MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
29:  epochs: 30


In [84]:

!find /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9 \
-type f -name "kaggle_val_split.csv"

/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/full_image_baseline_v2/kaggle_val_split.csv
/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/full_image_baseline_v3/kaggle_val_split.csv


In [85]:
!python /kaggle/working/train_baseline_v3.py \
    --config /kaggle/working/baseline_v3.yaml \
    --epochs 30 \
    --out_dir /kaggle/working

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
KAGGLE V3 TRAINING
Config: /kaggle/working/baseline_v3.yaml
Epochs: 30
2026-09-23 00:16:18,243 [INFO] Using device: cuda
Traceback (most recent call last):
  File "/kaggle/working/train_baseline_v3.py", line 594, in <module>
    main()
  File "/kaggle/working/train_baseline_v3.py", line 312, in main
    train_dataset = MAGFiLOPatchDataset(
                    ^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/src/data/dataset.py", line 85, in __init__
    self.parser = MAGFiLOAnnotationParser(json_path)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/src/data/annotations.py", line 7, in __init__
    with open(json_path, 'r') as f:
         ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/MAGFiLO_1.0_Kaggle_2026

In [86]:
import yaml

config_path = "/kaggle/working/baseline_v3.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

dataset_root = "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation"

config["dataset"]["image_dir"] = (
    dataset_root +
    "/MAGFiLO_1.0_Kaggle_2026/train/train_images"
)

config["dataset"]["json_path"] = (
    dataset_root +
    "/MAGFiLO_1.0_Kaggle_2026/train/"
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)

config["dataset"]["train_csv"] = (
    "/kaggle/working/outputs/kaggle_train_split.csv"
)

config["dataset"]["val_csv"] = (
    dataset_root +
    "/outputs/kaggle_val_split.csv"
)

config["training"]["epochs"] = 30

with open(config_path, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("V3 config updated.")
print("Image:", config["dataset"]["image_dir"])
print("JSON:", config["dataset"]["json_path"])
print("Train CSV:", config["dataset"]["train_csv"])
print("Val CSV:", config["dataset"]["val_csv"])
print("Epochs:", config["training"]["epochs"])

V3 config updated.
Image: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/train_images
JSON: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
Train CSV: /kaggle/working/outputs/kaggle_train_split.csv
Val CSV: /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv
Epochs: 30


In [87]:
import os

paths = [
    "/kaggle/working/outputs/kaggle_train_split.csv",
    "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv",
    "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/train_images",
    "/kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json",
]

for p in paths:
    print("OK" if os.path.exists(p) else "MISSING", p)

OK /kaggle/working/outputs/kaggle_train_split.csv
OK /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/outputs/kaggle_val_split.csv
OK /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/train_images
OK /kaggle/input/datasets/nytsoul/solar-filament-segmentation-9/solar-filament-segmentation/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json


In [88]:
!python /kaggle/working/train_baseline_v3.py \
    --config /kaggle/working/baseline_v3.yaml \
    --epochs 30 \
    --out_dir /kaggle/working

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
KAGGLE V3 TRAINING
Config: /kaggle/working/baseline_v3.yaml
Epochs: 30
2026-09-23 00:18:36,773 [INFO] Using device: cuda
2026-09-23 00:18:38,203 [INFO] Training patches per epoch: 2260 (565 images x 4 patches)
2026-09-23 00:18:38,203 [INFO] Validation images: 142 (full 2048x2048)
2026-09-23 00:18:38,650 [INFO] Initializing final segmentation_head bias to -4.5951 for prior 0.01
2026-09-23 00:18:39,331 [INFO] Initial model mean probability (on zero input): 0.0100
2026-09-23 00:18:39,362 [INFO] Using 2 GPUs for DataParallel
2026-09-23 00:18:39,675 [INFO] Loss config: Focal(alpha=0.25, gamma=2.0) + Tversky(alpha=0.7, beta=0.3)
2026-09-23 00:18:39,675 [INFO] AMP Status: True
2026-09-23 00:18:39,675 [INFO] Sampling probabilities: filament=0.50, disk=0.20, limb=0.15, background=0.15
2

In [89]:
!cp /kaggle/working/train_baseline_v3.py /kaggle/working/train_baseline_v3_ft.py

In [90]:
!grep -nE "model =|UnetPlusPlus|create_model|build_model|AdamW" /kaggle/working/train_baseline_v3_ft.py

349:    model = get_baseline_model(config)
365:        model = torch.nn.DataParallel(model)
368:    model = model.to(device)
370:    optimizer = torch.optim.AdamW(
502:        val_model = model.module if multi_gpu else model
560:    val_model = model.module if multi_gpu else model


In [92]:
!sed -n '340,380p' /kaggle/working/train_baseline_v3_ft.py

    train_loader = DataLoader(
        train_dataset,
        batch_size=config['training']['batch_size'],
        shuffle=True,
        num_workers=config['training'].get('num_workers', 4),
        pin_memory=(device.type == 'cuda')
    )

    # --- Model ---
    model = get_baseline_model(config)
    
    prior = config.get('model', {}).get('init_bias_prior', 0.01)
    bias_val = -math.log((1.0 - prior) / prior)
    logger.info(f"Initializing final segmentation_head bias to {bias_val:.4f} for prior {prior}")
    with torch.no_grad():
        model.segmentation_head[0].bias.fill_(bias_val)
        
    # Check initial probability on a dummy tensor
    dummy = torch.zeros(1, config['model'].get('in_channels', 1), 256, 256)
    initial_prob = torch.sigmoid(model(dummy)).mean().item()
    logger.info(f"Initial model mean probability (on zero input): {initial_prob:.4f}")

    multi_gpu = False
    if device.type == 'cuda' and config['training'].get('use_multi_gpu', False) and torch.cuda.d

In [93]:
import os

print("Best checkpoint:")
print(os.path.exists("/kaggle/working/checkpoints/best_baseline_v3.pth"))

print("\nCheckpoint files:")
!ls -lh /kaggle/working/checkpoints/

Best checkpoint:
True

Checkpoint files:
total 200M
-rw-r--r-- 1 root root 100M Sep 23 01:16 best_baseline_v3.pth
-rw-r--r-- 1 root root 100M Sep 23 03:33 latest_checkpoint_v3.pth


In [95]:
from pathlib import Path

src = Path("/kaggle/working/train_baseline_v3.py")
dst = Path("/kaggle/working/train_baseline_v3_resume.py")

text = src.read_text()

# ============================================================
# 1. Add resume argument
# ============================================================

old = """parser.add_argument('--epochs', type=int, default=None)
    parser.add_argument('--out_dir', type=str, default=None)"""

new = """parser.add_argument('--epochs', type=int, default=None)
    parser.add_argument('--resume', type=str, default=None,
                        help='Path to checkpoint to resume from')
    parser.add_argument('--out_dir', type=str, default=None)"""

text = text.replace(old, new)


# ============================================================
# 2. After model.to(device), load checkpoint
# ============================================================

old = """model = model.to(device)

    optimizer = torch.optim.AdamW("""

new = """model = model.to(device)

    # ========================================================
    # RESUME FROM EXISTING BEST V3 CHECKPOINT
    # ========================================================

    if args.resume:
        logger.info("=" * 60)
        logger.info("RESUMING V3 TRAINING")
        logger.info(f"Checkpoint: {args.resume}")

        if not os.path.exists(args.resume):
            raise FileNotFoundError(
                f"Resume checkpoint not found: {args.resume}"
            )

        checkpoint = torch.load(
            args.resume,
            map_location=device
        )

        # Handle DataParallel / normal model checkpoint
        if multi_gpu:
            model.module.load_state_dict(checkpoint)
        else:
            model.load_state_dict(checkpoint)

        logger.info("V3 checkpoint loaded successfully.")
        logger.info("=" * 60)

    optimizer = torch.optim.AdamW("""

text = text.replace(old, new)


# ============================================================
# 3. Change training epochs to 30 for this new run
# ============================================================

# Keep config at 30 but make command-line override possible.
# No change needed if --epochs 30 is supplied.


# ============================================================
# 4. Change experiment output directory
# ============================================================

text = text.replace(
    "full_image_baseline_v3",
    "full_image_baseline_v3_resume"
)

text = text.replace(
    "best_baseline_v3.pth",
    "best_baseline_v3_resume.pth"
)

text = text.replace(
    "latest_checkpoint_v3.pth",
    "latest_checkpoint_v3_resume.pth"
)

text = text.replace(
    "training_history_v3.csv",
    "training_history_v3_resume.csv"
)

text = text.replace(
    "training_history_v3.json",
    "training_history_v3_resume.json"
)

text = text.replace(
    "training_v3.log",
    "training_v3_resume.log"
)


dst.write_text(text)

print("Created:")
print(dst)

print("\nSize:")
print(dst.stat().st_size, "bytes")

Created:
/kaggle/working/train_baseline_v3_resume.py

Size:
24479 bytes


In [96]:
!grep -nE "resume|RESUMING|load_state_dict|v3_resume" /kaggle/working/train_baseline_v3_resume.py

220:    parser.add_argument('--resume', type=str, default=None,
221:                        help='Path to checkpoint to resume from')
266:    experiment_dir = os.path.join(out_dir_base, 'outputs', 'full_image_baseline_v3_resume')
272:    log_path = os.path.join(experiment_dir, 'training_v3_resume.log')
376:    if args.resume:
378:        logger.info("RESUMING V3 TRAINING")
379:        logger.info(f"Checkpoint: {args.resume}")
381:        if not os.path.exists(args.resume):
383:                f"Resume checkpoint not found: {args.resume}"
387:            args.resume,
393:            model.module.load_state_dict(checkpoint)
395:            model.load_state_dict(checkpoint)
565:            torch.save(model.state_dict(), os.path.join(checkpoints_dir, 'best_baseline_v3_resume.pth'))
568:        torch.save(model.state_dict(), os.path.join(checkpoints_dir, 'latest_checkpoint_v3_resume.pth'))
580:        pd.DataFrame(history_serializable).to_csv(os.path.join(experiment_dir, 'training_history_v

In [97]:
!python /kaggle/working/train_baseline_v3_resume.py \
    --config /kaggle/working/baseline_v3.yaml \
    --epochs 30 \
    --resume /kaggle/working/checkpoints/best_baseline_v3.pth

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
KAGGLE V3 TRAINING
Config: /kaggle/working/baseline_v3.yaml
Epochs: 30
2026-09-23 03:53:51,354 [INFO] Using device: cuda
2026-09-23 03:53:52,838 [INFO] Training patches per epoch: 2260 (565 images x 4 patches)
2026-09-23 03:53:52,838 [INFO] Validation images: 142 (full 2048x2048)
2026-09-23 03:53:53,244 [INFO] Initializing final segmentation_head bias to -4.5951 for prior 0.01
2026-09-23 03:53:53,841 [INFO] Initial model mean probability (on zero input): 0.0100
2026-09-23 03:53:53,876 [INFO] Using 2 GPUs for DataParallel
2026-09-23 03:53:54,201 [INFO] ============================================================
2026-09-23 03:53:54,201 [INFO] RESUMING V3 TRAINING
2026-09-23 03:53:54,202 [INFO] Checkpoint: /kaggle/working/checkpoints/best_baseline_v3.pth
Traceback (most recent ca

In [98]:
!sed -n '370,405p' /kaggle/working/train_baseline_v3_resume.py

    model = model.to(device)

    # ========================================================
    # RESUME FROM EXISTING BEST V3 CHECKPOINT
    # ========================================================

    if args.resume:
        logger.info("=" * 60)
        logger.info("RESUMING V3 TRAINING")
        logger.info(f"Checkpoint: {args.resume}")

        if not os.path.exists(args.resume):
            raise FileNotFoundError(
                f"Resume checkpoint not found: {args.resume}"
            )

        checkpoint = torch.load(
            args.resume,
            map_location=device
        )

        # Handle DataParallel / normal model checkpoint
        if multi_gpu:
            model.module.load_state_dict(checkpoint)
        else:
            model.load_state_dict(checkpoint)

        logger.info("V3 checkpoint loaded successfully.")
        logger.info("=" * 60)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['training']['learning_rate'],
 

In [100]:
if args.resume:
    logger.info("=" * 60)
    logger.info("RESUMING V3 TRAINING")
    logger.info(f"Checkpoint: {args.resume}")
    logger.info("=" * 60)

    if not os.path.exists(args.resume):
        raise FileNotFoundError(
            f"Resume checkpoint not found: {args.resume}"
        )

    checkpoint = torch.load(
        args.resume,
        map_location=device
    )

    # ------------------------------------------------------------
    # Handle DataParallel checkpoint
    # ------------------------------------------------------------
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        checkpoint = checkpoint["state_dict"]

    # Check whether checkpoint keys contain "module."
    first_key = next(iter(checkpoint.keys()))

    if first_key.startswith("module."):
        logger.info("Checkpoint contains DataParallel 'module.' prefix.")

        # Remove "module." prefix
        checkpoint = {
            key.replace("module.", "", 1): value
            for key, value in checkpoint.items()
        }

    # Load into the underlying model
    if multi_gpu:
        model.module.load_state_dict(checkpoint)
    else:
        model.load_state_dict(checkpoint)

    logger.info("Checkpoint loaded successfully.")

Old block was not found.


In [101]:
!sed -n '365,410p' /kaggle/working/train_baseline_v3_resume.py

    if device.type == 'cuda' and config['training'].get('use_multi_gpu', False) and torch.cuda.device_count() > 1:
        logger.info(f"Using {torch.cuda.device_count()} GPUs for DataParallel")
        model = torch.nn.DataParallel(model)
        multi_gpu = True

    model = model.to(device)

    # ========================================================
    # RESUME FROM EXISTING BEST V3 CHECKPOINT
    # ========================================================

    if args.resume:
        logger.info("=" * 60)
        logger.info("RESUMING V3 TRAINING")
        logger.info(f"Checkpoint: {args.resume}")

        if not os.path.exists(args.resume):
            raise FileNotFoundError(
                f"Resume checkpoint not found: {args.resume}"
            )

        checkpoint = torch.load(
            args.resume,
            map_location=device
        )

        # Handle DataParallel / normal model checkpoint
        if multi_gpu:
            model.module.load_state_dict(checkpoi

In [102]:
!grep -n -A35 -B5 "args.resume" /kaggle/working/train_baseline_v3_resume.py

371-
372-    # ========================================================
373-    # RESUME FROM EXISTING BEST V3 CHECKPOINT
374-    # ========================================================
375-
376:    if args.resume:
377-        logger.info("=" * 60)
378-        logger.info("RESUMING V3 TRAINING")
379:        logger.info(f"Checkpoint: {args.resume}")
380-
381:        if not os.path.exists(args.resume):
382-            raise FileNotFoundError(
383:                f"Resume checkpoint not found: {args.resume}"
384-            )
385-
386-        checkpoint = torch.load(
387:            args.resume,
388-            map_location=device
389-        )
390-
391-        # Handle DataParallel / normal model checkpoint
392-        if multi_gpu:
393-            model.module.load_state_dict(checkpoint)
394-        else:
395-            model.load_state_dict(checkpoint)
396-
397-        logger.info("V3 checkpoint loaded successfully.")
398-        logger.info("=" * 60)
399-
400-    optimizer = torch

In [106]:
# ============================================================
# REPLACE RESUME CHECKPOINT BLOCK
# ============================================================

file_path = "/kaggle/working/train_baseline_v3_resume.py"

with open(file_path, "r") as f:
    code = f.read()

old_block = '''    # ========================================================
    # RESUME FROM EXISTING BEST V3 CHECKPOINT
    # ========================================================

    if args.resume:
        logger.info("=" * 60)
        logger.info("RESUMING V3 TRAINING")
        logger.info(f"Checkpoint: {args.resume}")

        if not os.path.exists(args.resume):
            raise FileNotFoundError(
                f"Resume checkpoint not found: {args.resume}"
            )

        checkpoint = torch.load(
            args.resume,
            map_location=device
        )

        # Handle DataParallel / normal model checkpoint
        if multi_gpu:
            model.module.load_state_dict(checkpoint)
        else:
            model.load_state_dict(checkpoint)

        logger.info("V3 checkpoint loaded successfully.")
        logger.info("=" * 60)
'''

new_block = '''    # ========================================================
    # RESUME FROM EXISTING BEST V3 CHECKPOINT
    # ========================================================

    if args.resume:
        logger.info("=" * 60)
        logger.info("RESUMING V3 TRAINING")
        logger.info(f"Checkpoint: {args.resume}")

        if not os.path.exists(args.resume):
            raise FileNotFoundError(
                f"Resume checkpoint not found: {args.resume}"
            )

        checkpoint = torch.load(
            args.resume,
            map_location=device
        )

        # --------------------------------------------------------
        # Handle DataParallel checkpoint automatically
        # --------------------------------------------------------

        # Check whether checkpoint contains module.* keys
        first_key = next(iter(checkpoint.keys()))

        if first_key.startswith("module."):
            logger.info("Detected DataParallel checkpoint.")

            # Remove "module." prefix
            checkpoint = {
                key.replace("module.", "", 1): value
                for key, value in checkpoint.items()
            }

        # Load into underlying model
        if multi_gpu:
            model.module.load_state_dict(checkpoint)
        else:
            model.load_state_dict(checkpoint)

        logger.info("V3 checkpoint loaded successfully.")
        logger.info("=" * 60)
'''

if old_block not in code:
    print("Old block not found.")
    print("Searching for existing resume block...")

    start = code.find("    # ========================================================\n    # RESUME FROM EXISTING BEST V3 CHECKPOINT")

    if start == -1:
        raise RuntimeError("Could not find resume block in the file.")

    end = code.find("    optimizer = torch.optim.AdamW(", start)

    if end == -1:
        raise RuntimeError("Could not find optimizer block after resume section.")

    code = code[:start] + new_block + "\n" + code[end:]

else:
    code = code.replace(old_block, new_block)

with open(file_path, "w") as f:
    f.write(code)

print("✅ Resume checkpoint block replaced successfully.")

Old block not found.
Searching for existing resume block...
✅ Resume checkpoint block replaced successfully.


In [107]:
!sed -n '365,420p' /kaggle/working/train_baseline_v3_resume.py

    if device.type == 'cuda' and config['training'].get('use_multi_gpu', False) and torch.cuda.device_count() > 1:
        logger.info(f"Using {torch.cuda.device_count()} GPUs for DataParallel")
        model = torch.nn.DataParallel(model)
        multi_gpu = True

    model = model.to(device)

    # ========================================================
    # RESUME FROM EXISTING BEST V3 CHECKPOINT
    # ========================================================

    if args.resume:
        logger.info("=" * 60)
        logger.info("RESUMING V3 TRAINING")
        logger.info(f"Checkpoint: {args.resume}")

        if not os.path.exists(args.resume):
            raise FileNotFoundError(
                f"Resume checkpoint not found: {args.resume}"
            )

        checkpoint = torch.load(
            args.resume,
            map_location=device
        )

        # --------------------------------------------------------
        # Handle DataParallel checkpoint automatically
     

In [109]:
!ls -lh /kaggle/working/checkpoints/best_baseline_v3.pth

-rw-r--r-- 1 root root 100M Sep 23 01:16 /kaggle/working/checkpoints/best_baseline_v3.pth


In [110]:
!python /kaggle/working/train_baseline_v3_resume.py \
    --resume /kaggle/working/checkpoints/best_baseline_v3.pth \
    --epochs 30

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
KAGGLE V3 TRAINING
Config: /kaggle/working/baseline_v3.yaml
Epochs: 30
2026-09-23 04:00:53,387 [INFO] Using device: cuda
2026-09-23 04:00:54,669 [INFO] Training patches per epoch: 2260 (565 images x 4 patches)
2026-09-23 04:00:54,669 [INFO] Validation images: 142 (full 2048x2048)
2026-09-23 04:00:55,077 [INFO] Initializing final segmentation_head bias to -4.5951 for prior 0.01
2026-09-23 04:00:55,551 [INFO] Initial model mean probability (on zero input): 0.0100
2026-09-23 04:00:55,583 [INFO] Using 2 GPUs for DataParallel
2026-09-23 04:00:55,843 [INFO] ============================================================
2026-09-23 04:00:55,843 [INFO] RESUMING V3 TRAINING
2026-09-23 04:00:55,843 [INFO] Checkpoint: /kaggle/working/checkpoints/best_baseline_v3.pth
2026-09-23 04:00:55,988 [